# LeetCode 312 — Burst Balloons (Hard) bằng Python

Notebook này giảng nhập môn bài toán (link: https://leetcode.com/problems/burst-balloons/):

```python
class Solution:
    def maxCoins(self, nums: List[int]) -> int:

```

**Mục tiêu sau khi học xong:**

1. Hiểu đúng luật chơi: "hàng xóm" là quả bóng **còn sống** đứng kề bên **tại thời điểm nổ**, nên **thứ tự nổ** quyết định số xu; tự tay tính được `[3,1,5,8] → 167`.
2. Biết **mô phỏng** một thứ tự nổ bằng `list.pop`, rồi viết **Cách 1 — brute force** thử mọi thứ tự nổ bằng `itertools.permutations` (n! thứ tự).
3. Học **đệ quy (recursion)** và **memoization (ghi nhớ kết quả)** từ căn bản; viết **Cách 2 — đệ quy + memo trên tuple các bóng còn lại** (2^n trạng thái) và thấy vì sao vẫn chưa đủ.
4. Hiểu vì sao nghĩ theo bóng nổ **ĐẦU TIÊN** không chia được bài toán thành hai bài con độc lập, và **bước đột phá**: nghĩ theo bóng nổ **CUỐI CÙNG** `k` trong một **khoảng mở** `(i, j)` có hai "bức tường".
5. Viết **Cách 3 — interval DP top-down** (đệ quy + memo trên cặp `(i, j)`) và **Cách 4 — DP bảng bottom-up theo độ dài khoảng** (cách chính), in bảng `dp` và các lựa chọn `k` cho `[3,1,5,8]`.
6. **Cách 5 — truy vết**: dựng lại **thứ tự nổ tối ưu** từ bảng lựa chọn, rồi kiểm chứng bằng chính hàm mô phỏng.
7. Nộp `class Solution`, kiểm thử chéo mọi cách với brute force (n ≤ 7), và nhận ra pattern **interval DP "chọn phần tử cuối cùng làm vách ngăn"** để dùng lại cho các bài tương tự.

## 1. Đề bài

Cho `n` quả bóng bay, đánh chỉ số từ `0` đến `n - 1`. Mỗi quả bóng được sơn một con số, biểu diễn bởi mảng `nums`. Bạn được yêu cầu làm nổ **tất cả** các quả bóng.

Nếu làm nổ quả bóng thứ `i`, bạn nhận được `nums[i - 1] * nums[i] * nums[i + 1]` đồng xu (coins). Nếu `i - 1` hoặc `i + 1` vượt ra ngoài mảng, hãy coi như ở đó có một quả bóng được sơn số `1`.

Trả về **số xu lớn nhất** có thể thu được bằng cách làm nổ các quả bóng một cách khôn ngoan.

**Ví dụ 1**

```text
Input: nums = [3,1,5,8]
Output: 167
Explanation:
nums = [3,1,5,8] --> [3,5,8] --> [3,8] --> [8] --> []
coins =  3*1*5    +   3*5*8   +  1*3*8  + 1*8*1 = 167
```

**Ví dụ 2**

```text
Input: nums = [1,5]
Output: 10
```

**Constraints (ràng buộc):**

- `n == nums.length`
- `1 <= n <= 300`
- `0 <= nums[i] <= 100`

### Đọc đề chậm lại: chỗ nào dễ hiểu sai?

- `nums[i - 1]` và `nums[i + 1]` trong đề là **hai quả bóng còn sống đang đứng kề** quả `i` **tại lúc nổ**, không phải hai hàng xóm ban đầu. Khi một quả bóng nổ, nó biến mất, hai quả ở hai bên nó **dịch lại gần nhau** và trở thành hàng xóm. Nhìn Ví dụ 1: sau khi nổ `1`, mảng còn `[3,5,8]`, nên lúc nổ `5` hàng xóm của nó là `3` và `8` → được `3*5*8 = 120`.
- Phải nổ **hết** `n` quả. Thứ duy nhất ta được chọn là **thứ tự nổ**. Với `n` quả có `n!` (n giai thừa) thứ tự khác nhau.
- Hai quả bóng "ảo" số `1` ở ngoài hai đầu **không bao giờ nổ** và không cho xu — chúng chỉ để **nhân vào** khi quả bóng ở mép nổ.
- `nums[i]` có thể bằng `0`; `n` có thể bằng `1` (khi đó đáp án là `1 * nums[0] * 1 = nums[0]`).
- Giới hạn `n ≤ 300` là một **gợi ý về độ phức tạp**: thuật toán cỡ `O(n^3)` (khoảng 4,5 triệu phép thử với `n = 300`) là vừa sức; còn bất cứ thứ gì **hàm mũ** (`2^n`, `n!`) đều không thể chạy xong.

Cell dưới lưu 2 ví dụ của đề để dùng lại cho mọi cách giải phía sau.

In [1]:
EXAMPLES = [
    ([3, 1, 5, 8], 167),
    ([1, 5], 10),
]

for nums, expected in EXAMPLES:
    print(f"nums = {nums} -> kỳ vọng {expected}   (n = {len(nums)})")

nums = [3, 1, 5, 8] -> kỳ vọng 167   (n = 4)
nums = [1, 5] -> kỳ vọng 10   (n = 2)


### Tính tay Ví dụ 1 — và thử một thứ tự khác

Trước khi viết bất cứ dòng code nào, ta tính tay để "cảm" được luật chơi. Thứ tự trong lời giải của đề là: nổ `1` → nổ `5` → nổ `3` → nổ `8` (theo **chỉ số gốc**: `1, 2, 0, 3`).

| Bước | Mảng trước khi nổ | Nổ bóng (chỉ số gốc) | Hàng xóm trái | Hàng xóm phải | Xu | Tổng |
|:---:|---|:---:|:---:|:---:|---:|---:|
| 1 | `[3, 1, 5, 8]` | `1` (chỉ số 1) | 3 | 5 | `3*1*5 = 15` | 15 |
| 2 | `[3, 5, 8]` | `5` (chỉ số 2) | 3 | 8 | `3*5*8 = 120` | 135 |
| 3 | `[3, 8]` | `3` (chỉ số 0) | 1 (ảo) | 8 | `1*3*8 = 24` | 159 |
| 4 | `[8]` | `8` (chỉ số 3) | 1 (ảo) | 1 (ảo) | `1*8*1 = 8` | **167** |

Bây giờ thử thứ tự "ngây thơ" nhất: nổ **từ trái sang phải** (chỉ số `0, 1, 2, 3`):

| Bước | Mảng trước khi nổ | Nổ bóng | Hàng xóm trái | Hàng xóm phải | Xu | Tổng |
|:---:|---|:---:|:---:|:---:|---:|---:|
| 1 | `[3, 1, 5, 8]` | `3` | 1 (ảo) | 1 | `1*3*1 = 3` | 3 |
| 2 | `[1, 5, 8]` | `1` | 1 (ảo) | 5 | `1*1*5 = 5` | 8 |
| 3 | `[5, 8]` | `5` | 1 (ảo) | 8 | `1*5*8 = 40` | 48 |
| 4 | `[8]` | `8` | 1 (ảo) | 1 (ảo) | `1*8*1 = 8` | **56** |

Cùng 4 quả bóng, thứ tự khác nhau cho `167` và `56` — chênh gấp 3 lần. Quan sát thứ tự tốt: hai quả lớn `3` và `8` được **giữ lại lâu** để làm "hệ số nhân" cho các quả khác, và quả lớn nhất `8` nổ **cuối cùng**. Ghi nhớ quan sát "quả nổ cuối cùng" này — nó chính là chìa khóa của lời giải tối ưu ở mục 5.

## 2. Kiến thức nền

### 2.1. `list.pop(i)`: xóa một phần tử và các phần tử phía sau "dịch lại gần"

Trong Python, `a.pop(i)` làm 2 việc:

1. **Trả về** phần tử đang ở **vị trí** `i`.
2. **Xóa** nó khỏi list; mọi phần tử phía sau **lùi lên 1 vị trí**.

Ví dụ `a = [3, 1, 5, 8]`, gọi `a.pop(1)` → trả về `1`, còn lại `a = [3, 5, 8]`. Hai quả `3` và `5` giờ đứng cạnh nhau — chính là hiện tượng "hàng xóm thay đổi" của đề. Đồng thời quả `5` **đổi vị trí** từ 2 xuống 1: vị trí trong list **không cố định**, đây là chỗ hay gây nhầm khi mô phỏng (mục 2.3 sẽ xử lý).

In [2]:
a = [3, 1, 5, 8]
print("Ban đầu      :", a, "| quả 5 đang ở vị trí", a.index(5))
removed = a.pop(1)                     # nổ quả ở VỊ TRÍ 1
print("pop(1) trả về:", removed)
print("Sau khi pop  :", a, "| quả 5 giờ ở vị trí", a.index(5))
p = a.index(5)
print("Hàng xóm hiện tại của 5:", a[p - 1], "và", a[p + 1])

Ban đầu      : [3, 1, 5, 8] | quả 5 đang ở vị trí 2
pop(1) trả về: 1
Sau khi pop  : [3, 5, 8] | quả 5 giờ ở vị trí 1
Hàng xóm hiện tại của 5: 3 và 8


### 2.2. Lấy hàng xóm với "biên 1 ảo" — và cái bẫy chỉ số âm

Giả sử danh sách bóng **hiện tại** là `cur` (dài `L`) và ta nổ quả ở vị trí `p`:

- Hàng xóm trái: `cur[p - 1]` nếu `p > 0`, ngược lại là bóng ảo `1`.
- Hàng xóm phải: `cur[p + 1]` nếu `p + 1 < L`, ngược lại là bóng ảo `1`.
- Xu nhận được: `trái * cur[p] * phải`.

**Bẫy:** trong Python, chỉ số âm **không báo lỗi** — `cur[-1]` là phần tử **cuối cùng**. Nếu lười kiểm tra và viết thẳng `cur[p - 1]` khi `p = 0`, Python sẽ lặng lẽ lấy phần tử cuối làm hàng xóm trái → sai mà không hề có thông báo lỗi. Vì vậy phải kiểm tra `p > 0` một cách tường minh.

In [3]:
def coins_at(cur, p):
    # Số xu nhận được nếu nổ quả bóng ở VỊ TRÍ p của danh sách hiện tại cur.
    left = cur[p - 1] if p > 0 else 1
    right = cur[p + 1] if p + 1 < len(cur) else 1
    return left * cur[p] * right

cur = [3, 1, 5, 8]
for p in range(len(cur)):
    print(f"nổ vị trí {p} (bóng {cur[p]}) -> {coins_at(cur, p)} xu")
print("Bẫy chỉ số âm: cur[0 - 1] =", cur[0 - 1], "(là phần tử CUỐI, không phải bóng ảo 1!)")

nổ vị trí 0 (bóng 3) -> 3 xu
nổ vị trí 1 (bóng 1) -> 15 xu
nổ vị trí 2 (bóng 5) -> 40 xu
nổ vị trí 3 (bóng 8) -> 40 xu
Bẫy chỉ số âm: cur[0 - 1] = 8 (là phần tử CUỐI, không phải bóng ảo 1!)


### 2.3. Mô phỏng một thứ tự nổ cho trước

Ta muốn viết hàm `simulate_order(nums, order)`: nổ bóng theo thứ tự `order` rồi trả về tổng xu. `order` ghi theo **chỉ số gốc**, ví dụ thứ tự của đề là `[1, 2, 0, 3]`.

Vấn đề: sau mỗi lần `pop`, **vị trí** của các quả bóng thay đổi (mục 2.1), nên không thể dùng chỉ số gốc như vị trí. Cách xử lý:

- Giữ list `alive` chứa **chỉ số gốc** của các quả còn sống, ban đầu `[0, 1, 2, 3]`.
- Muốn nổ quả có chỉ số gốc `idx`: tìm **vị trí hiện tại** `p = alive.index(idx)`, lấy hàng xóm `alive[p - 1]`, `alive[p + 1]` (hoặc bóng ảo 1), cộng xu, rồi `alive.pop(p)`.

Ví dụ: `alive = [0, 1, 2, 3]`, nổ `idx = 1` → `p = 1`, hàng xóm là chỉ số 0 và 2 (giá trị 3 và 5) → 15 xu; sau đó `alive = [0, 2, 3]`. Nổ tiếp `idx = 2` → giờ `p = 1` (không còn là 2!), hàng xóm là chỉ số 0 và 3 (giá trị 3 và 8) → 120 xu.

Vì sao lưu **chỉ số** chứ không lưu **giá trị**? Vì giá trị có thể trùng nhau (ví dụ `[1, 1, 1]`), còn chỉ số gốc thì luôn khác nhau, nên `alive.index(idx)` luôn tìm đúng quả cần nổ.

In [4]:
def simulate_order(nums, order, verbose=False):
    # Nổ bóng theo thứ tự `order` (chỉ số GỐC), trả về tổng xu.
    alive = list(range(len(nums)))          # chỉ số gốc của các bóng còn sống
    total = 0
    for step, idx in enumerate(order, start=1):
        p = alive.index(idx)                # vị trí HIỆN TẠI của bóng idx
        left = nums[alive[p - 1]] if p > 0 else 1
        right = nums[alive[p + 1]] if p + 1 < len(alive) else 1
        gain = left * nums[idx] * right
        total += gain
        if verbose:
            print(f"  Bước {step}: còn {[nums[t] for t in alive]}, nổ chỉ số {idx} (bóng {nums[idx]}): "
                  f"{left}*{nums[idx]}*{right} = {gain:>3} | tổng = {total}")
        alive.pop(p)
    return total

print("Thứ tự của đề [1, 2, 0, 3]:")
print("=> Tổng:", simulate_order([3, 1, 5, 8], [1, 2, 0, 3], verbose=True))
print("Trái sang phải [0, 1, 2, 3]:")
print("=> Tổng:", simulate_order([3, 1, 5, 8], [0, 1, 2, 3], verbose=True))

Thứ tự của đề [1, 2, 0, 3]:
  Bước 1: còn [3, 1, 5, 8], nổ chỉ số 1 (bóng 1): 3*1*5 =  15 | tổng = 15
  Bước 2: còn [3, 5, 8], nổ chỉ số 2 (bóng 5): 3*5*8 = 120 | tổng = 135
  Bước 3: còn [3, 8], nổ chỉ số 0 (bóng 3): 1*3*8 =  24 | tổng = 159
  Bước 4: còn [8], nổ chỉ số 3 (bóng 8): 1*8*1 =   8 | tổng = 167
=> Tổng: 167
Trái sang phải [0, 1, 2, 3]:
  Bước 1: còn [3, 1, 5, 8], nổ chỉ số 0 (bóng 3): 1*3*1 =   3 | tổng = 3
  Bước 2: còn [1, 5, 8], nổ chỉ số 1 (bóng 1): 1*1*5 =   5 | tổng = 8
  Bước 3: còn [5, 8], nổ chỉ số 2 (bóng 5): 1*5*8 =  40 | tổng = 48
  Bước 4: còn [8], nổ chỉ số 3 (bóng 8): 1*8*1 =   8 | tổng = 56
=> Tổng: 56


### 2.4. `itertools.permutations`: liệt kê mọi thứ tự

`permutations(range(n))` sinh **tất cả** các cách sắp xếp (hoán vị — permutation) của `0, 1, ..., n-1`, mỗi cách là một `tuple`. Với `n` phần tử có `n! = 1 * 2 * ... * n` hoán vị: `3! = 6`, `4! = 24`, `8! = 40320`.

Hàm `math.factorial(n)` tính `n!`. Hai công cụ này đủ để viết cách "thử hết" đầu tiên.

In [5]:
from itertools import permutations
from math import factorial

for order in permutations(range(3)):
    print(order)
print("Số hoán vị của 3 phần tử:", len(list(permutations(range(3)))), "= 3! =", factorial(3))
print("4! =", factorial(4), "| 8! =", factorial(8), "| 12! =", factorial(12))

(0, 1, 2)
(0, 2, 1)
(1, 0, 2)
(1, 2, 0)
(2, 0, 1)
(2, 1, 0)
Số hoán vị của 3 phần tử: 6 = 3! = 6
4! = 24 | 8! = 40320 | 12! = 479001600


## 3. Cách 1 — Brute force: thử mọi thứ tự nổ

Đây là cách người mới tự nghĩ ra đầu tiên: *"Không biết thứ tự nào tốt nhất? Thử hết!"*

```text
best = 0
với mỗi thứ tự order trong permutations(range(n)):
    best = max(best, simulate_order(nums, order))
trả về best
```

Cách này **chắc chắn đúng** vì nó xét đúng định nghĩa của đề: mọi thứ tự có thể đều được thử, nên giá trị lớn nhất không thể bị bỏ sót. Nó là "trọng tài" để kiểm tra các cách thông minh hơn phía sau.

Cell dưới còn in **bảng xếp hạng** 24 thứ tự của `[3, 1, 5, 8]` để ta quan sát thứ tự tốt trông như thế nào.

In [6]:
def max_coins_bruteforce(nums):
    best = 0
    for order in permutations(range(len(nums))):
        best = max(best, simulate_order(nums, order))
    return best

for nums, expected in EXAMPLES:
    print(nums, "->", max_coins_bruteforce(nums), "(kỳ vọng", expected, ")")

nums = [3, 1, 5, 8]
ranking = sorted(((simulate_order(nums, o), o) for o in permutations(range(4))), reverse=True)
print("\nTop 5 thứ tự của [3, 1, 5, 8]:")
for coins, order in ranking[:5]:
    print(f"  {order} -> {coins:>3} xu, bóng nổ CUỐI = {nums[order[-1]]}")
print("3 thứ tự tệ nhất:", [(o, c) for c, o in ranking[-3:]])

[3, 1, 5, 8] -> 167 (kỳ vọng 167 )
[1, 5] -> 10 (kỳ vọng 10 )

Top 5 thứ tự của [3, 1, 5, 8]:
  (1, 2, 0, 3) -> 167 xu, bóng nổ CUỐI = 8
  (1, 2, 3, 0) -> 162 xu, bóng nổ CUỐI = 3
  (2, 1, 0, 3) ->  96 xu, bóng nổ CUỐI = 8
  (2, 1, 3, 0) ->  91 xu, bóng nổ CUỐI = 3
  (1, 0, 2, 3) ->  78 xu, bóng nổ CUỐI = 8
3 thứ tự tệ nhất: [((3, 2, 0, 1), 49), ((3, 0, 2, 1), 49), ((0, 3, 2, 1), 49)]


### Cách 1 vướng ở đâu?

Nhìn bảng xếp hạng: chỉ **một** thứ tự duy nhất đạt 167, và các thứ tự tốt đều để quả lớn (`8` hoặc `3`) nổ **cuối**. Nhưng với input bất kỳ ta không thể "đoán" thứ tự — buộc phải tìm kiếm.

Vấn đề là số thứ tự `n!` tăng **khủng khiếp**:

- `n = 8` → 40.320 thứ tự; `n = 10` → 3.628.800; `n = 20` → khoảng `2,4 * 10^18`.
- Mỗi thứ tự còn phải mô phỏng `n` lần nổ, mỗi lần `alive.index` và `pop` tốn `O(n)`.

**Độ phức tạp Cách 1** (n = `len(nums)`): Time `O(n! * n^2)`, Space `O(n)`. Với `n = 300` là vô vọng. Đo thử:

In [7]:
import time, random

random.seed(0)
for n in [6, 7, 8]:
    nums = [random.randint(0, 100) for _ in range(n)]
    t0 = time.perf_counter()
    max_coins_bruteforce(nums)
    print(f"n={n}: {factorial(n):>6} thứ tự, mất {time.perf_counter() - t0:.3f} giây")
print("300! có", len(str(factorial(300))), "chữ số -> không bao giờ chạy xong.")

n=6:    720 thứ tự, mất 0.001 giây
n=7:   5040 thứ tự, mất 0.010 giây
n=8:  40320 thứ tự, mất 0.063 giây
300! có 615 chữ số -> không bao giờ chạy xong.


## 4. Cách 2 — Đệ quy + memo trên "các bóng còn lại"

Trước khi cải tiến, ta cần hai công cụ: **đệ quy** và **memoization**.

### 4.1. Kiến thức nền: đệ quy (recursion)

**Đệ quy** là khi một hàm **tự gọi lại chính nó** trên một phiên bản **nhỏ hơn** của bài toán. Một hàm đệ quy luôn có 2 phần:

1. **Base case (trường hợp cơ sở):** bài toán đủ nhỏ để trả lời ngay, không cần gọi tiếp. Thiếu phần này hàm sẽ gọi mãi không dừng.
2. **Bước đệ quy:** giải bài lớn bằng cách gọi hàm trên bài nhỏ hơn rồi ghép kết quả.

Ví dụ: tổng một list. `total([3, 1, 5]) = 3 + total([1, 5])`, `total([1, 5]) = 1 + total([5])`, ..., `total([]) = 0` (base case). Cell dưới in ra từng tầng lời gọi (thụt lề theo độ sâu) để thấy máy tính "đi xuống" tới base case rồi "đi lên" ghép kết quả.

In [8]:
def total_rec(a, depth=0):
    pad = "    " * depth
    if not a:                                   # base case: list rỗng
        print(pad + "total([]) = 0   <- base case, bắt đầu đi lên")
        return 0
    print(pad + f"total({a}) = {a[0]} + total({a[1:]})")
    result = a[0] + total_rec(a[1:], depth + 1)  # bước đệ quy trên list ngắn hơn
    print(pad + f"=> total({a}) = {result}")
    return result

print("Kết quả:", total_rec([3, 1, 5]))

total([3, 1, 5]) = 3 + total([1, 5])
    total([1, 5]) = 1 + total([5])
        total([5]) = 5 + total([])
            total([]) = 0   <- base case, bắt đầu đi lên
        => total([5]) = 5
    => total([1, 5]) = 6
=> total([3, 1, 5]) = 9
Kết quả: 9


### 4.2. Memoization: đừng giải lại bài con đã giải

Nhiều hàm đệ quy gọi **cùng một bài con rất nhiều lần**. Ví dụ dãy Fibonacci: `fib(n) = fib(n-1) + fib(n-2)`. Khi tính `fib(5)`, `fib(3)` được tính 2 lần, `fib(2)` 3 lần... số lời gọi tăng theo hàm mũ.

**Memoization** (ghi nhớ) = dùng một "cuốn sổ" (thường là `dict`) lưu kết quả bài con:

1. Trước khi giải bài con `x`: nếu `x` đã có trong sổ → trả kết quả ngay.
2. Nếu chưa: giải, **ghi vào sổ**, rồi trả về.

Nhờ vậy mỗi bài con chỉ được giải **một lần**. Đếm số lời gọi để thấy khác biệt:

In [9]:
calls = {"plain": 0, "memo": 0}

def fib_plain(n):
    calls["plain"] += 1
    if n <= 1:
        return n
    return fib_plain(n - 1) + fib_plain(n - 2)

memo = {}
def fib_memo(n):
    calls["memo"] += 1
    if n <= 1:
        return n
    if n in memo:                                  # 1) đã giải rồi -> trả ngay
        return memo[n]
    memo[n] = fib_memo(n - 1) + fib_memo(n - 2)    # 2) giải lần đầu -> ghi vào sổ
    return memo[n]

print("fib(20) =", fib_plain(20), "| số lời gọi KHÔNG memo:", calls["plain"])
print("fib(20) =", fib_memo(20), "| số lời gọi CÓ memo   :", calls["memo"])

fib(20) = 6765 | số lời gọi KHÔNG memo: 21891
fib(20) = 6765 | số lời gọi CÓ memo   : 39


### 4.3. `functools.lru_cache` — và vì sao cần `tuple`

Viết sổ `memo` bằng tay hơi dài. Python có sẵn **decorator** `@lru_cache(maxsize=None)` (đặt ngay trên dòng `def`): nó tự động ghi nhớ kết quả theo **bộ tham số** của hàm. `cache_info()` cho biết số lần trúng sổ (`hits`), số lần phải tính (`misses`) và số mục đang lưu (`currsize`).

**Điều kiện quan trọng:** tham số phải **hashable** (dùng được làm key của `dict`): `int`, `str`, `tuple` thì được; **`list` thì không** (vì list có thể bị sửa sau khi đã làm key). Vậy nếu muốn ghi nhớ theo "danh sách các bóng còn lại", ta phải biểu diễn nó bằng **`tuple`**.

Bỏ phần tử ở vị trí `p` khỏi tuple `rem` (tuple không có `pop`): ghép hai lát cắt `rem[:p] + rem[p + 1:]` — phần trước `p` và phần sau `p`.

In [10]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n):
    return n if n <= 1 else fib_cached(n - 1) + fib_cached(n - 2)

print("fib(80) =", fib_cached(80), "|", fib_cached.cache_info())

rem = (3, 1, 5, 8)
p = 1
print("Bỏ vị trí", p, "khỏi", rem, "->", rem[:p] + rem[p + 1:])

cache = {}
cache[(3, 5, 8)] = 999                 # tuple làm key: OK
try:
    cache[[3, 5, 8]] = 999             # list làm key: lỗi
except TypeError as e:
    print("list làm key bị lỗi:", e)

fib(80) = 23416728348467685 | CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)
Bỏ vị trí 1 khỏi (3, 1, 5, 8) -> (3, 5, 8)
list làm key bị lỗi: unhashable type: 'list'


### 4.4. Ý tưởng Cách 2: bài con = "các bóng còn lại"

Brute force lãng phí ở đâu? Thứ tự `(0, 1, ...)` và `(1, 0, ...)` của `[3, 1, 5, 8]`: nổ `3` rồi `1`, hay nổ `1` rồi `3`, thì **đều còn lại `[5, 8]`**. Từ thời điểm đó, số xu thu thêm **chỉ phụ thuộc vào các bóng còn lại** (và thứ tự trái → phải của chúng), không phụ thuộc quá khứ. Vậy phần "nổ hết `[5, 8]` tốt nhất" chỉ cần tính **một lần**.

Định nghĩa: `g(rem)` = số xu lớn nhất khi phải nổ hết các bóng trong tuple `rem` (hai đầu ngoài là bóng ảo 1).

- Base case: `g(()) = 0` (không còn bóng).
- Bước đệ quy: thử nổ **trước** quả ở mỗi vị trí `p`:
  `g(rem) = max over p của [ coins_at(rem, p) + g(rem bỏ vị trí p) ]`.

Ví dụ: `g((5, 8))`: nổ `5` trước được `1*5*8 = 40`, còn `g((8,)) = 8` → 48; nổ `8` trước được `5*8*1 = 40`, còn `g((5,)) = 5` → 45. Vậy `g((5, 8)) = 48`.

Số trạng thái: mỗi quả hoặc còn hoặc đã nổ → tối đa `2^n` tuple khác nhau. So với `n!`: `n = 10` chỉ có `1024` trạng thái thay vì 3,6 triệu thứ tự.

In [11]:
def max_coins_subset_memo(nums, return_states=False):
    @lru_cache(maxsize=None)
    def g(rem):                          # rem: tuple các bóng còn lại (trái -> phải)
        if not rem:
            return 0
        best = 0
        for p in range(len(rem)):        # thử nổ quả ở vị trí p TRƯỚC
            best = max(best, coins_at(rem, p) + g(rem[:p] + rem[p + 1:]))
        return best

    ans = g(tuple(nums))
    if return_states:
        return ans, g.cache_info().currsize   # số trạng thái đã ghi nhớ (kể cả tuple rỗng)
    return ans

for nums, expected in EXAMPLES:
    print(nums, "->", max_coins_subset_memo(nums), "(kỳ vọng", expected, ")")
print("g((5, 8)) =", max_coins_subset_memo([5, 8]))

[3, 1, 5, 8] -> 167 (kỳ vọng 167 )
[1, 5] -> 10 (kỳ vọng 10 )
g((5, 8)) = 48


### Trace Cách 2 với `nums = [3, 1, 5]`

Hàm `trace_subset_memo` in một dòng **mỗi khi một trạng thái được tính xong lần đầu** (thứ tự "con trước, cha sau" — vì cha cần kết quả của con), kèm lựa chọn tốt nhất: nổ quả nào **trước** rồi giải phần còn lại. Lần gặp lại một trạng thái đã có trong sổ thì không in, chỉ đếm số lần dùng lại.

In [12]:
def trace_subset_memo(nums):
    memo = {}
    stats = {"hit": 0}

    def g(rem):
        if not rem:
            return 0
        if rem in memo:
            stats["hit"] += 1
            return memo[rem]
        best, best_p = -1, None
        for p in range(len(rem)):
            val = coins_at(rem, p) + g(rem[:p] + rem[p + 1:])
            if val > best:
                best, best_p = val, p
        memo[rem] = best
        rest = rem[:best_p] + rem[best_p + 1:]
        print(f"g{str(rem):<10} = {best:>3}   tốt nhất: nổ {rem[best_p]} TRƯỚC "
              f"({coins_at(rem, best_p)} xu) rồi giải g{rest}")
        return best

    ans = g(tuple(nums))
    print(f"Đáp án = {ans}; số trạng thái đã lưu = {len(memo)}; số lần dùng lại memo = {stats['hit']}")

trace_subset_memo([3, 1, 5])

g(5,)       =   5   tốt nhất: nổ 5 TRƯỚC (5 xu) rồi giải g()
g(1,)       =   1   tốt nhất: nổ 1 TRƯỚC (1 xu) rồi giải g()
g(1, 5)     =  10   tốt nhất: nổ 1 TRƯỚC (5 xu) rồi giải g(5,)
g(3,)       =   3   tốt nhất: nổ 3 TRƯỚC (3 xu) rồi giải g()
g(3, 5)     =  20   tốt nhất: nổ 3 TRƯỚC (15 xu) rồi giải g(5,)
g(3, 1)     =   6   tốt nhất: nổ 1 TRƯỚC (3 xu) rồi giải g(3,)
g(3, 1, 5)  =  35   tốt nhất: nổ 1 TRƯỚC (15 xu) rồi giải g(3, 5)
Đáp án = 35; số trạng thái đã lưu = 7; số lần dùng lại memo = 3


### Đọc trace Cách 2 — và Cách 2 vướng ở đâu?

Dòng cuối cùng `g(3, 1, 5) = 35` nói: tốt nhất là nổ `1` trước (`3*1*5 = 15` xu), còn lại `(3, 5)` có giá trị `g(3, 5) = 20` → `15 + 20 = 35`. Có 7 trạng thái khác rỗng = `2^3 - 1`, và 3 lần dùng lại sổ (các tuple 1 phần tử `(5,)`, `(1,)`, `(3,)` được nhiều đường đi tới).

Chú ý: Cách 2 **đúng** dù nó nghĩ theo bóng nổ **đầu tiên**, vì nó giữ **nguyên cả tuple** còn lại làm trạng thái — không cố tách thành hai nửa. Cái giá phải trả là số trạng thái:

- Mỗi quả còn/mất → tới `2^n` tuple. `n = 20` → khoảng 1 triệu; `n = 30` → 1 tỉ; `n = 300` → `2^300 ≈ 10^90` (nhiều hơn số nguyên tử trong vũ trụ quan sát được).
- Mỗi trạng thái thử `≤ n` vị trí, mỗi lần tạo tuple mới tốn `O(n)`.

**Độ phức tạp Cách 2:** Time `O(2^n * n^2)`, Space `O(2^n * n)` (mỗi key trong sổ là một tuple dài tới `n`). Tốt hơn `n!` rất nhiều, nhưng **vẫn là hàm mũ**. Đo thử (dùng giá trị khác nhau để không trạng thái nào bị trùng):

In [13]:
print(" n |             n! |    2^n | trạng thái thực tế | thời gian")
for n in [8, 10, 12, 14, 16]:
    nums = random.sample(range(1, 101), n)
    t0 = time.perf_counter()
    ans, states = max_coins_subset_memo(nums, return_states=True)
    print(f"{n:>2} | {factorial(n):>14} | {2 ** n:>6} | {states:>18} | {time.perf_counter() - t0:.3f}s")

 n |             n! |    2^n | trạng thái thực tế | thời gian
 8 |          40320 |    256 |                256 | 0.000s
10 |        3628800 |   1024 |               1024 | 0.002s
12 |      479001600 |   4096 |               4096 | 0.011s


14 |    87178291200 |  16384 |              16384 | 0.046s


16 | 20922789888000 |  65536 |              65536 | 0.212s


## 5. Tìm bài con "đẹp": một đoạn liên tiếp

Muốn thoát khỏi hàm mũ, trạng thái phải được mô tả bằng thứ gì đó **nhỏ hơn nhiều** so với "một tập con bất kỳ". Ứng viên lý tưởng: một **đoạn liên tiếp** của mảng, mô tả bằng 2 số `(i, j)` — chỉ có khoảng `n^2 / 2` đoạn (với `n = 300` là ~45.000, quá ít!). Câu hỏi: làm sao để bài con luôn là một đoạn liên tiếp?

### 5.1. Thử nghĩ theo bóng nổ ĐẦU TIÊN — vì sao không chia được

Cách chia quen thuộc của nhiều bài DP: *"chọn quả `k` nổ đầu tiên, nhận `nums[k-1] * nums[k] * nums[k+1]`, rồi giải **riêng** nửa trái `nums[0..k-1]` và nửa phải `nums[k+1..n-1]`"*.

Nghe hợp lý, nhưng **sai**: ngay sau khi `k` nổ, quả `k-1` (cuối nửa trái) và quả `k+1` (đầu nửa phải) **trở thành hàng xóm**. Sau này khi nổ `k-1`, số xu của nó phụ thuộc vào việc `k+1` (thuộc nửa phải) **còn sống hay đã nổ**. Hai nửa "dính" vào nhau, nên không thể giải độc lập rồi cộng lại.

Ví dụ cụ thể với `[3, 1, 5, 8]`: nổ `1` trước → còn `[3, 5, 8]`, "nửa trái" là `[3]`, "nửa phải" là `[5, 8]`. Xu của quả `3` (nửa trái) lại phụ thuộc nửa phải đã nổ tới đâu:

In [14]:
print("Xu của bóng 3 (nửa trái) phụ thuộc vào việc nửa phải đã nổ tới đâu:")
for right_alive in ([5, 8], [8], []):
    cur = [3] + right_alive
    print(f"  nửa phải còn {str(right_alive):<7} -> nổ 3 được {coins_at(cur, 0):>2} xu")

Xu của bóng 3 (nửa trái) phụ thuộc vào việc nửa phải đã nổ tới đâu:
  nửa phải còn [5, 8]  -> nổ 3 được 15 xu
  nửa phải còn [8]     -> nổ 3 được 24 xu
  nửa phải còn []      -> nổ 3 được  3 xu


Cùng là "nổ quả 3", nhưng được 15, 24 hoặc 3 xu tùy **bên kia** đang ra sao → nửa trái **không có giá trị riêng** → không chia được.

### 5.2. Bước đột phá: nghĩ theo bóng nổ CUỐI CÙNG

Đảo ngược câu hỏi: thay vì hỏi "quả nào nổ **đầu tiên**?", hãy hỏi **"quả nào nổ CUỐI CÙNG?"**.

**Bước 1 — Đệm biên.** Tạo `arr = [1] + nums + [1]` (dài `m = n + 2`). Quả thật nằm ở chỉ số `1..n`; `arr[0]` và `arr[n+1]` là hai bóng ảo **không bao giờ nổ**.

**Bước 2 — Định nghĩa bài con trên khoảng MỞ.**
`dp(i, j)` = số xu lớn nhất khi nổ **hết các quả nằm giữa** `i` và `j` (tức các chỉ số `k` với `i < k < j`), với điều kiện **quả `i` và quả `j` vẫn còn nguyên** suốt quá trình — chúng là hai **bức tường**.

**Bước 3 — Chọn quả nổ cuối cùng `k` trong `(i, j)`.**

1. Suốt thời gian trước khi `k` nổ, `k` **luôn đứng đó**. Mọi quả bên trái `k` chỉ có hàng xóm nằm trong đoạn `[i..k]`; chúng **không bao giờ "nhìn thấy"** bên phải `k`. Tương tự cho bên phải. → Hai nhóm **thực sự độc lập**, và mỗi nhóm lại đúng là bài con cùng dạng: `dp(i, k)` (tường `i`, `k`) và `dp(k, j)` (tường `k`, `j`).
2. Khi `k` nổ (cuối cùng), mọi quả ở giữa đã hết → hàng xóm của `k` **chính là hai bức tường** `i` và `j` → được `arr[i] * arr[k] * arr[j]` xu.

**Công thức:**

```text
dp(i, j) = max over i < k < j của  dp(i, k) + arr[i] * arr[k] * arr[j] + dp(k, j)
dp(i, j) = 0                  nếu j - i < 2   (không có quả nào ở giữa)
Đáp án   = dp(0, n + 1)
```

**Vì sao "cuối cùng" lại chữa được bệnh của "đầu tiên"?** Nổ `k` đầu tiên **xóa** bức vách giữa hai nhóm, làm chúng dính vào nhau. Chọn `k` nổ cuối cùng thì `k` **là** bức vách đứng vững tới phút chót, tách hai nhóm ra. Câu nên nhớ: *"Ai sống lâu nhất thì làm tường."*

Ví dụ `[3, 1, 5, 8]`: `arr = [1, 3, 1, 5, 8, 1]`, bài toán là `dp(0, 5)`. Nếu chọn `k = 4` (quả `8`) nổ cuối: quả 8 được `1*8*1 = 8`; nhóm trái là `dp(0, 4)` = nổ hết `3, 1, 5` giữa hai tường `1` và `8`; nhóm phải `dp(4, 5)` rỗng = 0. Cell dưới liệt kê cả 4 lựa chọn `k`:

In [15]:
nums = [3, 1, 5, 8]
arr = [1] + nums + [1]
print("arr =", arr, "| chỉ số:", list(range(len(arr))))
i, j = 0, len(arr) - 1
for k in range(i + 1, j):
    print(f"k={k} (bóng {arr[k]}) nổ cuối: trái = dp({i},{k}) với bóng {str(arr[i+1:k]):<9} "
          f"phải = dp({k},{j}) với bóng {str(arr[k+1:j]):<9} k được {arr[i]}*{arr[k]}*{arr[j]} = {arr[i]*arr[k]*arr[j]}")

arr = [1, 3, 1, 5, 8, 1] | chỉ số: [0, 1, 2, 3, 4, 5]
k=1 (bóng 3) nổ cuối: trái = dp(0,1) với bóng []        phải = dp(1,5) với bóng [1, 5, 8] k được 1*3*1 = 3
k=2 (bóng 1) nổ cuối: trái = dp(0,2) với bóng [3]       phải = dp(2,5) với bóng [5, 8]    k được 1*1*1 = 1
k=3 (bóng 5) nổ cuối: trái = dp(0,3) với bóng [3, 1]    phải = dp(3,5) với bóng [8]       k được 1*5*1 = 5
k=4 (bóng 8) nổ cuối: trái = dp(0,4) với bóng [3, 1, 5] phải = dp(4,5) với bóng []        k được 1*8*1 = 8


### 5.3. Tính tay vài ô nhỏ — và kiểm chứng định nghĩa "có tường"

Bài con `dp(i, j)` có hai bức tường **không nhất thiết bằng 1**. Ví dụ `dp(0, 3)` với `arr = [1, 3, 1, 5, 8, 1]`: nổ hết hai quả `3, 1` (chỉ số 1, 2) giữa tường `arr[0] = 1` và `arr[3] = 5`.

- **Khoảng chứa 1 quả** (`j = i + 2`): chỉ có một lựa chọn `k = i + 1`, nên `dp(i, i+2) = arr[i] * arr[i+1] * arr[i+2]`. Ví dụ `dp(1, 3) = 3*1*5 = 15`, `dp(0, 2) = 1*3*1 = 3`.
- **`dp(0, 3)`**, hai lựa chọn:
  - `k = 1` (quả `3` nổ cuối): `dp(0,1) + 1*3*5 + dp(1,3) = 0 + 15 + 15 = 30`.
  - `k = 2` (quả `1` nổ cuối): `dp(0,2) + 1*1*5 + dp(2,3) = 3 + 5 + 0 = 8`.
  - → `dp(0, 3) = 30`.

Để chắc chắn ta hiểu đúng ý nghĩa của `dp(i, j)`, viết một brute force nhỏ **có tường**: thử mọi thứ tự nổ các quả bên trong, hai tường cố định không nổ. Nếu định nghĩa đúng, kết quả phải khớp với số tính tay (30) và với các ô khác của bảng mà ta sẽ thấy ở mục 7.

In [16]:
def brute_with_walls(left_wall, inner, right_wall):
    # Brute force: nổ hết các quả trong `inner`; hai bức tường không bao giờ nổ.
    best = 0
    for order in permutations(range(len(inner))):
        alive = list(range(len(inner)))
        total = 0
        for idx in order:
            p = alive.index(idx)
            left = inner[alive[p - 1]] if p > 0 else left_wall
            right = inner[alive[p + 1]] if p + 1 < len(alive) else right_wall
            total += left * inner[idx] * right
            alive.pop(p)
        best = max(best, total)
    return best

print("dp(0,3): bóng [3, 1]    giữa tường 1 và 5 ->", brute_with_walls(1, [3, 1], 5))
print("dp(1,4): bóng [1, 5]    giữa tường 3 và 8 ->", brute_with_walls(3, [1, 5], 8))
print("dp(0,4): bóng [3, 1, 5] giữa tường 1 và 8 ->", brute_with_walls(1, [3, 1, 5], 8))
print("dp(0,5): toàn bộ bài toán (tường 1 và 1)  ->", brute_with_walls(1, [3, 1, 5, 8], 1))

dp(0,3): bóng [3, 1]    giữa tường 1 và 5 -> 30
dp(1,4): bóng [1, 5]    giữa tường 3 và 8 -> 135
dp(0,4): bóng [3, 1, 5] giữa tường 1 và 8 -> 159
dp(0,5): toàn bộ bài toán (tường 1 và 1)  -> 167


## 6. Cách 3 — Interval DP top-down (đệ quy + memo trên `(i, j)`)

**Interval DP** (quy hoạch động trên khoảng) là kiểu DP mà mỗi trạng thái là một **đoạn/khoảng** `(i, j)` của mảng, và đoạn lớn được giải từ các đoạn nhỏ hơn nằm bên trong nó.

Cách 3 dịch **nguyên văn** công thức ở mục 5.2 thành hàm đệ quy, rồi gắn `@lru_cache` để mỗi cặp `(i, j)` chỉ tính một lần. Trạng thái giờ chỉ là **hai số nguyên** — hashable, nhỏ gọn, và chỉ có `O(n^2)` cặp.

Từng dòng trong code:

- `arr = [1] + nums + [1]`: đệm hai bức tường ảo.
- `if j - i < 2: return 0`: base case — giữa `i` và `j` không có quả nào (`j = i + 1`).
- `for k in range(i + 1, j)`: `k` chạy qua **mọi quả nằm giữa** (không gồm `i`, `j` vì chúng là tường).
- `dp(i, k) + arr[i] * arr[k] * arr[j] + dp(k, j)`: nhóm trái + xu của quả cuối + nhóm phải.
- Đáp án là `dp(0, len(arr) - 1)` = `dp(0, n + 1)`.

In [17]:
def max_coins_topdown(nums):
    arr = [1] + nums + [1]                   # thêm 2 bức tường ảo

    @lru_cache(maxsize=None)
    def dp(i, j):
        if j - i < 2:                        # không có quả nào nằm giữa i và j
            return 0
        best = 0
        for k in range(i + 1, j):            # k = quả nổ CUỐI CÙNG trong (i, j)
            best = max(best, dp(i, k) + arr[i] * arr[k] * arr[j] + dp(k, j))
        return best

    return dp(0, len(arr) - 1)

for nums, expected in EXAMPLES:
    print(nums, "->", max_coins_topdown(nums), "(kỳ vọng", expected, ")")
print("[5] ->", max_coins_topdown([5]), "| [0, 0] ->", max_coins_topdown([0, 0]))

[3, 1, 5, 8] -> 167 (kỳ vọng 167 )
[1, 5] -> 10 (kỳ vọng 10 )
[5] -> 5 | [0, 0] -> 0


### Trace Cách 3: cây lời gọi cho `nums = [3, 1, 5]`

Với `arr = [1, 3, 1, 5, 1]`, bài toán là `dp(0, 4)`. Hàm `trace_topdown` in:

- `dp(i,j): bóng [...] giữa tường a|b` khi **bắt đầu** tính một trạng thái mới (thụt lề theo độ sâu đệ quy);
- `=> dp(i,j) = ...` khi tính **xong**, kèm giá trị của từng lựa chọn `k` và `k` được chọn;
- `đã có trong memo` khi gặp lại một trạng thái đã tính.

Các base case (khoảng rỗng, giá trị 0) không in cho đỡ rối.

In [18]:
def trace_topdown(nums):
    arr = [1] + nums + [1]
    memo = {}

    def dp(i, j, depth):
        pad = "    " * depth
        if j - i < 2:
            return 0
        if (i, j) in memo:
            print(f"{pad}dp({i},{j}) đã có trong memo = {memo[(i, j)]}")
            return memo[(i, j)]
        print(f"{pad}dp({i},{j}): bóng {arr[i+1:j]} giữa tường {arr[i]}|{arr[j]}")
        best, best_k, parts = -1, None, []
        for k in range(i + 1, j):
            val = dp(i, k, depth + 1) + arr[i] * arr[k] * arr[j] + dp(k, j, depth + 1)
            parts.append(f"k={k}:{val}")
            if val > best:
                best, best_k = val, k
        memo[(i, j)] = best
        print(f"{pad}=> dp({i},{j}) = {best}  [{' | '.join(parts)}] chọn k={best_k} (bóng {arr[best_k]} nổ cuối)")
        return best

    print("Kết quả:", dp(0, len(arr) - 1, 0))

trace_topdown([3, 1, 5])

dp(0,4): bóng [3, 1, 5] giữa tường 1|1
    dp(1,4): bóng [1, 5] giữa tường 3|1
        dp(2,4): bóng [5] giữa tường 1|1
        => dp(2,4) = 5  [k=3:5] chọn k=3 (bóng 5 nổ cuối)
        dp(1,3): bóng [1] giữa tường 3|5
        => dp(1,3) = 15  [k=2:15] chọn k=2 (bóng 1 nổ cuối)
    => dp(1,4) = 30  [k=2:8 | k=3:30] chọn k=3 (bóng 5 nổ cuối)
    dp(0,2): bóng [3] giữa tường 1|1
    => dp(0,2) = 3  [k=1:3] chọn k=1 (bóng 3 nổ cuối)
    dp(2,4) đã có trong memo = 5
    dp(0,3): bóng [3, 1] giữa tường 1|5
        dp(1,3) đã có trong memo = 15
        dp(0,2) đã có trong memo = 3
    => dp(0,3) = 30  [k=1:30 | k=2:8] chọn k=1 (bóng 3 nổ cuối)
=> dp(0,4) = 35  [k=1:33 | k=2:9 | k=3:35] chọn k=3 (bóng 5 nổ cuối)
Kết quả: 35


### Đọc trace bằng bảng

Các trạng thái theo thứ tự **tính xong** (`arr = [1, 3, 1, 5, 1]`):

| Trạng thái | Bóng ở giữa | Tường | Các lựa chọn `k` → giá trị | Kết quả |
|---|---|:---:|---|---:|
| `dp(2,4)` | `[5]` | 1 \| 1 | `k=3`: `0 + 1*5*1 + 0 = 5` | 5 |
| `dp(1,3)` | `[1]` | 3 \| 5 | `k=2`: `0 + 3*1*5 + 0 = 15` | 15 |
| `dp(1,4)` | `[1, 5]` | 3 \| 1 | `k=2`: `0 + 3*1*1 + 5 = 8`; `k=3`: `15 + 3*5*1 + 0 = 30` | 30 |
| `dp(0,2)` | `[3]` | 1 \| 1 | `k=1`: `0 + 1*3*1 + 0 = 3` | 3 |
| `dp(0,3)` | `[3, 1]` | 1 \| 5 | `k=1`: `0 + 1*3*5 + 15 = 30`; `k=2`: `3 + 1*1*5 + 0 = 8` | 30 |
| `dp(0,4)` | `[3, 1, 5]` | 1 \| 1 | `k=1`: `0+3+30 = 33`; `k=2`: `3+1+5 = 9`; `k=3`: `30+5+0 = 35` | **35** |

Đáp án 35 ứng với `k = 3`: quả `5` nổ **cuối**; trong nhóm trái `dp(0,3)` thì quả `3` nổ cuối; còn lại quả `1` nổ đầu tiên → thứ tự `1, 3, 5` cho `3*1*5 + 1*3*5 + 1*5*1 = 15 + 15 + 5 = 35` — khớp với kết quả `g(3, 1, 5) = 35` của Cách 2.

Ba dòng `đã có trong memo` là lúc memo tiết kiệm công: `dp(2,4)`, `dp(1,3)`, `dp(0,2)` được nhiều trạng thái cha cần đến.

### Độ phức tạp Cách 3

Gọi `n = len(nums)`, `m = n + 2`.

- **Số trạng thái** `(i, j)` với `0 ≤ i < j ≤ n + 1`: khoảng `m^2 / 2` → `O(n^2)`.
- **Mỗi trạng thái** thử tối đa `n` giá trị `k`. Chính xác hơn, tổng số lần thử là số bộ ba `i < k < j` chọn từ `m` điểm = `C(n+2, 3)`; với `n = 300` là `302*301*300/6 = 4.545.100`.
- → **Time `O(n^3)`**, **Space `O(n^2)`** cho sổ memo, cộng **độ sâu đệ quy `O(n)`** (chuỗi `dp(0,n+1) → dp(0,n) → ...`). Python mặc định cho đệ quy sâu khoảng 1000 tầng nên `n = 300` vẫn chạy được, nhưng đây là điểm yếu so với bản bottom-up ở mục sau.

## 7. Cách 4 — DP bảng bottom-up theo độ dài khoảng (cách chính)

Top-down "đi từ bài lớn xuống bài nhỏ" nhờ đệ quy. **Bottom-up** làm ngược lại: tự tay **điền bảng** từ bài nhỏ nhất lên bài lớn nhất bằng vòng lặp. Ưu điểm: không đệ quy (không lo giới hạn độ sâu), không tốn chi phí gọi hàm, và **in được cả bảng** để quan sát.

### Kiến thức nền: bảng 2 chiều (list của list)

Bảng `dp` có `m` hàng, `m` cột; `dp[i][j]` là ô ở **hàng `i`, cột `j`**. Cách tạo đúng:

```python
dp = [[0] * m for _ in range(m)]   # m hàng, mỗi hàng là một list MỚI
```

Cách tạo **sai** rất hay gặp: `[[0] * m] * m` — nó tạo `m` **tham chiếu tới cùng một hàng**, sửa một ô là cả cột đổi theo:

In [19]:
m = 3
good = [[0] * m for _ in range(m)]
bad = [[0] * m] * m
good[0][2] = 7
bad[0][2] = 7
print("good:", good)
print("bad :", bad, "<- cả 3 hàng đổi theo vì là CÙNG một list!")
print("bad[0] is bad[1]:", bad[0] is bad[1])

good: [[0, 0, 7], [0, 0, 0], [0, 0, 0]]
bad : [[0, 0, 7], [0, 0, 7], [0, 0, 7]] <- cả 3 hàng đổi theo vì là CÙNG một list!
bad[0] is bad[1]: True


### Thứ tự điền bảng: theo độ dài khoảng tăng dần

Muốn tính `dp[i][j]`, ta cần `dp[i][k]` và `dp[k][j]` với `i < k < j`. Cả hai đều là khoảng **ngắn hơn** (`k - i < j - i` và `j - k < j - i`). Vậy nếu điền bảng theo **độ dài khoảng** `length = j - i` **tăng dần**, mọi ô cần dùng luôn có sẵn.

Với `n = 4` (`m = 6`):

```text
length = 1: (0,1) (1,2) (2,3) (3,4) (4,5)   <- khoảng rỗng, dp = 0 sẵn, không cần tính
length = 2: (0,2) (1,3) (2,4) (3,5)         <- mỗi khoảng chứa 1 quả
length = 3: (0,3) (1,4) (2,5)               <- 2 quả
length = 4: (0,4) (1,5)                     <- 3 quả
length = 5: (0,5)                           <- cả 4 quả = ĐÁP ÁN (góc trên bên phải bảng)
```

Trên bảng, ta điền **từng đường chéo**, từ sát đường chéo chính ra dần góc trên bên phải.

**Khung vòng lặp và các cận:**

```python
for length in range(2, m):           # length = j - i, từ 2 (1 quả) tới m - 1 (cả bài)
    for i in range(0, m - length):   # i lớn nhất là m - 1 - length để j = i + length <= m - 1
        j = i + length
        for k in range(i + 1, j):    # mọi quả nằm GIỮA i và j
            ...
```

Code chính dưới đây ghép khung này với công thức của mục 5.2. Biến `best` giữ giá trị lớn nhất trong các lựa chọn `k`; vì mọi số xu đều `≥ 0` nên khởi tạo `best = 0` là an toàn.

In [20]:
def max_coins_dp(nums):
    arr = [1] + nums + [1]
    m = len(arr)
    dp = [[0] * m for _ in range(m)]           # dp[i][j] = 0 khi j - i < 2
    for length in range(2, m):                 # độ dài khoảng j - i, từ ngắn tới dài
        for i in range(0, m - length):
            j = i + length
            best = 0
            for k in range(i + 1, j):          # k = quả nổ cuối cùng trong (i, j)
                coins = dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j]
                if coins > best:
                    best = coins
            dp[i][j] = best
    return dp[0][m - 1]

for nums, expected in EXAMPLES:
    print(nums, "->", max_coins_dp(nums), "(kỳ vọng", expected, ")")

[3, 1, 5, 8] -> 167 (kỳ vọng 167 )
[1, 5] -> 10 (kỳ vọng 10 )


### Trace Cách 4: in bảng `dp` cho `[3, 1, 5, 8]`

Hàm `trace_dp` chạy đúng các vòng lặp của `max_coins_dp`, nhưng với mỗi ô `dp[i][j]` nó in: các quả nằm giữa, giá trị của **từng** lựa chọn `k` (dạng `k=...:giá trị`), và `k` được chọn. Cuối cùng in toàn bộ bảng (dấu `.` = ô với `j ≤ i`, không có nghĩa).

In [21]:
def trace_dp(nums):
    arr = [1] + nums + [1]
    m = len(arr)
    dp = [[0] * m for _ in range(m)]
    print("arr =", arr, "(quả thật ở chỉ số 1 ..", m - 2, ")")
    for length in range(2, m):
        print(f"--- length = {length}: mỗi khoảng chứa {length - 1} quả ---")
        for i in range(0, m - length):
            j = i + length
            parts, best, best_k = [], -1, None
            for k in range(i + 1, j):
                coins = dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j]
                parts.append(f"k={k}:{coins}")
                if coins > best:
                    best, best_k = coins, k
            dp[i][j] = best
            print(f"  dp[{i}][{j}] bóng {str(arr[i+1:j]):<13} {' | '.join(parts):<34}"
                  f" -> k={best_k} (bóng {arr[best_k]}) = {best}")
    print("Bảng dp cuối (hàng i, cột j):")
    print("      " + "".join(f"j={j:<5}" for j in range(m)))
    for i in range(m):
        print(f"i={i}:  " + "".join(f"{(dp[i][j] if j > i else '.'):<7}" for j in range(m)))
    return dp[0][m - 1]

print("Đáp án dp[0][5] =", trace_dp([3, 1, 5, 8]))

arr = [1, 3, 1, 5, 8, 1] (quả thật ở chỉ số 1 .. 4 )
--- length = 2: mỗi khoảng chứa 1 quả ---
  dp[0][2] bóng [3]           k=1:3                              -> k=1 (bóng 3) = 3
  dp[1][3] bóng [1]           k=2:15                             -> k=2 (bóng 1) = 15
  dp[2][4] bóng [5]           k=3:40                             -> k=3 (bóng 5) = 40
  dp[3][5] bóng [8]           k=4:40                             -> k=4 (bóng 8) = 40
--- length = 3: mỗi khoảng chứa 2 quả ---
  dp[0][3] bóng [3, 1]        k=1:30 | k=2:8                     -> k=1 (bóng 3) = 30
  dp[1][4] bóng [1, 5]        k=2:64 | k=3:135                   -> k=3 (bóng 5) = 135
  dp[2][5] bóng [5, 8]        k=3:45 | k=4:48                    -> k=4 (bóng 8) = 48
--- length = 4: mỗi khoảng chứa 3 quả ---
  dp[0][4] bóng [3, 1, 5]     k=1:159 | k=2:51 | k=3:70          -> k=1 (bóng 3) = 159
  dp[1][5] bóng [1, 5, 8]     k=2:51 | k=3:70 | k=4:159          -> k=4 (bóng 8) = 159
--- length = 5: mỗi khoảng chứa 4 quả ---
  d

### Đọc trace bằng bảng

`arr = [1, 3, 1, 5, 8, 1]`:

| Ô | Bóng ở giữa | Tường | Lựa chọn `k` → giá trị | Chọn | `dp` |
|---|---|:---:|---|:---:|---:|
| `dp[0][2]` | `[3]` | 1 \| 1 | `k=1`: 3 | 1 | 3 |
| `dp[1][3]` | `[1]` | 3 \| 5 | `k=2`: 15 | 2 | 15 |
| `dp[2][4]` | `[5]` | 1 \| 8 | `k=3`: 40 | 3 | 40 |
| `dp[3][5]` | `[8]` | 5 \| 1 | `k=4`: 40 | 4 | 40 |
| `dp[0][3]` | `[3, 1]` | 1 \| 5 | `k=1`: 30; `k=2`: 8 | 1 | 30 |
| `dp[1][4]` | `[1, 5]` | 3 \| 8 | `k=2`: 64; `k=3`: 135 | 3 | 135 |
| `dp[2][5]` | `[5, 8]` | 1 \| 1 | `k=3`: 45; `k=4`: 48 | 4 | 48 |
| `dp[0][4]` | `[3, 1, 5]` | 1 \| 8 | `k=1`: 159; `k=2`: 51; `k=3`: 70 | 1 | 159 |
| `dp[1][5]` | `[1, 5, 8]` | 3 \| 1 | `k=2`: 51; `k=3`: 70; `k=4`: 159 | 4 | 159 |
| `dp[0][5]` | `[3, 1, 5, 8]` | 1 \| 1 | `k=1`: 162; `k=2`: 52; `k=3`: 75; `k=4`: 167 | 4 | **167** |

Kiểm tra một ô bằng tay — `dp[1][4]` (hai quả `1, 5` giữa tường `3` và `8`):

- `k = 2` (quả `1` nổ cuối): `dp[1][2] + 3*1*8 + dp[2][4] = 0 + 24 + 40 = 64`.
- `k = 3` (quả `5` nổ cuối): `dp[1][3] + 3*5*8 + dp[3][4] = 15 + 120 + 0 = 135`. → chọn 135.

Và ô đáp án `dp[0][5]` với `k = 4` (quả `8` nổ cuối): `dp[0][4] + 1*8*1 + dp[4][5] = 159 + 8 + 0 = 167`.

**Đọc ý nghĩa vài ô:** `dp[0][4] = 159` = "nổ hết `3, 1, 5` khi hai bên là tường `1` và `8`" — khớp với brute force có tường ở mục 5.3; `dp[2][5] = 48` = "nổ hết `5, 8` giữa hai tường 1" — khớp `g((5, 8)) = 48` của Cách 2. Các con số "gặp nhau" từ ba hướng khác nhau là dấu hiệu tốt rằng ta hiểu đúng định nghĩa.

### Độ phức tạp Cách 4

Ba vòng lặp lồng nhau: `length` (≤ n+1 giá trị) × `i` (≤ n) × `k` (≤ n), tổng chính xác là `C(n+2, 3)` lần thử `k` như Cách 3.

- **Time `O(n^3)`** — với `n = 300` khoảng 4,5 triệu phép thử, Python chạy dưới 1 giây.
- **Space `O(n^2)`** — bảng `(n+2) × (n+2)` ≈ 91.000 ô với `n = 300`.

## 8. Cách 5 — Truy vết: dựng lại thứ tự nổ tối ưu

Bảng `dp` cho biết **bao nhiêu** xu, nhưng chưa nói **nổ theo thứ tự nào**. Ý tưởng truy vết (backtracking the choices): khi điền bảng, lưu thêm `choice[i][j] = k` tốt nhất. Sau đó dựng thứ tự bằng đệ quy:

```text
build(i, j):
    nếu j - i < 2: trả về []                       (khoảng rỗng)
    k = choice[i][j]                               (quả nổ CUỐI trong (i, j))
    trả về build(i, k) + build(k, j) + [k - 1]
```

- `build(i, k)` rồi `build(k, j)`: nổ hết nhóm trái theo thứ tự tối ưu của nó, rồi nhóm phải — trong lúc đó `k` vẫn đứng làm tường, nên số xu đúng như `dp[i][k]` và `dp[k][j]`.
- `[k - 1]` đặt **cuối**: `k` nổ sau cùng. Trừ 1 để đổi chỉ số trong `arr` (đã đệm) về **chỉ số gốc** trong `nums`.

Một chi tiết nhỏ khi lưu `choice`: nếu mọi lựa chọn đều cho 0 xu (ví dụ `nums = [0, 0]`), điều kiện `coins > dp[i][j]` không bao giờ đúng và `choice` sẽ kẹt ở `None`. Vì vậy ta viết `choice[i][j] is None or coins > dp[i][j]` để luôn có ít nhất một `k`.

Hàm `build_dp_tables` dưới đây điền cả hai bảng; `max_coins_with_order` dùng nó để trả về `(số xu, thứ tự nổ)`. Ta kiểm chứng bằng cách đưa thứ tự vào `simulate_order` (mục 2.3): nếu truy vết đúng, mô phỏng phải ra đúng bằng `dp`.

In [22]:
def build_dp_tables(nums):
    arr = [1] + nums + [1]
    m = len(arr)
    dp = [[0] * m for _ in range(m)]
    choice = [[None] * m for _ in range(m)]
    for length in range(2, m):
        for i in range(0, m - length):
            j = i + length
            for k in range(i + 1, j):
                coins = dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j]
                if choice[i][j] is None or coins > dp[i][j]:
                    dp[i][j] = coins
                    choice[i][j] = k
    return arr, dp, choice

def max_coins_with_order(nums):
    arr, dp, choice = build_dp_tables(nums)

    def build(i, j):
        if j - i < 2:
            return []
        k = choice[i][j]
        return build(i, k) + build(k, j) + [k - 1]   # k - 1: chỉ số arr -> chỉ số nums

    return dp[0][len(arr) - 1], build(0, len(arr) - 1)

for nums in ([3, 1, 5, 8], [1, 5], [7, 9, 8, 0, 7], [0, 0]):
    best, order = max_coins_with_order(nums)
    print(f"{str(nums):<16} max = {best:>4} | thứ tự nổ (chỉ số gốc) = {str(order):<16}"
          f" | mô phỏng lại = {simulate_order(nums, order)}")

[3, 1, 5, 8]     max =  167 | thứ tự nổ (chỉ số gốc) = [1, 2, 0, 3]     | mô phỏng lại = 167
[1, 5]           max =   10 | thứ tự nổ (chỉ số gốc) = [0, 1]           | mô phỏng lại = 10
[7, 9, 8, 0, 7]  max = 1001 | thứ tự nổ (chỉ số gốc) = [3, 2, 1, 4, 0]  | mô phỏng lại = 1001
[0, 0]           max =    0 | thứ tự nổ (chỉ số gốc) = [1, 0]           | mô phỏng lại = 0


### Vẽ "cây quả nổ cuối"

Bảng `choice` thực chất mô tả một **cây**: gốc là quả nổ cuối cùng của cả bài; con trái là cây của khoảng bên trái nó, con phải là cây của khoảng bên phải. Đọc cây **từ lá lên gốc** chính là thứ tự nổ.

Với `[3, 1, 5, 8]`, thứ tự truy vết được là `[1, 2, 0, 3]` — **trùng khớp** thứ tự trong lời giải của đề (nổ `1` → `5` → `3` → `8`). Cây của nó là một "chuỗi" (mỗi nút chỉ có một nhánh không rỗng) nên chỉ có đúng một thứ tự tối ưu — khớp bảng xếp hạng ở Cách 1.

Ví dụ thứ hai `[1, 2, 9, 2, 1]`: quả `9` nổ cuối và **cả hai bên đều có bóng**. Nhóm trái và nhóm phải độc lập, nên ta có thể **xen kẽ** các lần nổ của hai nhóm theo nhiều cách mà tổng xu không đổi → có **nhiều** thứ tự tối ưu (Bài 6 sẽ thách bạn đếm chúng).

In [23]:
def print_burst_tree(nums):
    arr, dp, choice = build_dp_tables(nums)

    def show(i, j, depth):
        if j - i < 2:
            return
        k = choice[i][j]
        print("    " * depth + f"({i},{j}) bóng {arr[i+1:j]}: nổ CUỐI bóng {arr[k]} (chỉ số gốc {k - 1})"
              f" được {arr[i]}*{arr[k]}*{arr[j]} = {arr[i] * arr[k] * arr[j]}")
        show(i, k, depth + 1)
        show(k, j, depth + 1)

    show(0, len(arr) - 1, 0)
    best, order = max_coins_with_order(nums)
    print("=> max =", best, "| một thứ tự tối ưu:", order)

print_burst_tree([3, 1, 5, 8])
print()
print_burst_tree([1, 2, 9, 2, 1])

(0,5) bóng [3, 1, 5, 8]: nổ CUỐI bóng 8 (chỉ số gốc 3) được 1*8*1 = 8
    (0,4) bóng [3, 1, 5]: nổ CUỐI bóng 3 (chỉ số gốc 0) được 1*3*8 = 24
        (1,4) bóng [1, 5]: nổ CUỐI bóng 5 (chỉ số gốc 2) được 3*5*8 = 120
            (1,3) bóng [1]: nổ CUỐI bóng 1 (chỉ số gốc 1) được 3*1*5 = 15
=> max = 167 | một thứ tự tối ưu: [1, 2, 0, 3]

(0,6) bóng [1, 2, 9, 2, 1]: nổ CUỐI bóng 9 (chỉ số gốc 2) được 1*9*1 = 9
    (0,3) bóng [1, 2]: nổ CUỐI bóng 1 (chỉ số gốc 0) được 1*1*9 = 9
        (1,3) bóng [2]: nổ CUỐI bóng 2 (chỉ số gốc 1) được 1*2*9 = 18
    (3,6) bóng [2, 1]: nổ CUỐI bóng 1 (chỉ số gốc 4) được 9*1*1 = 9
        (3,5) bóng [2]: nổ CUỐI bóng 2 (chỉ số gốc 3) được 9*2*1 = 18
=> max = 63 | một thứ tự tối ưu: [1, 0, 3, 4, 2]


## 9. Code nộp LeetCode

Ta nộp bản **bottom-up** (Cách 4) vì: không đệ quy, nhanh, dễ kiểm soát bộ nhớ. Hai tối ưu nhỏ cho Python (không đổi độ phức tạp, chỉ giảm hằng số):

- `edge = arr[i] * arr[j]` tính **một lần** cho mỗi ô thay vì lặp lại trong vòng `k`.
- `row_i = dp[i]` lưu tham chiếu tới hàng `i`, để `row_i[k]` khỏi phải tra `dp[i]` mỗi lần.

`from typing import List` cần khi chạy ngoài LeetCode (trên LeetCode đã có sẵn).

In [24]:
from typing import List

class Solution:
    def maxCoins(self, nums: List[int]) -> int:
        arr = [1] + nums + [1]
        m = len(arr)
        dp = [[0] * m for _ in range(m)]
        for length in range(2, m):
            for i in range(m - length):
                j = i + length
                edge = arr[i] * arr[j]
                row_i = dp[i]
                best = 0
                for k in range(i + 1, j):
                    coins = row_i[k] + edge * arr[k] + dp[k][j]
                    if coins > best:
                        best = coins
                row_i[j] = best
        return dp[0][m - 1]


sol = Solution()
tests = [
    ([3, 1, 5, 8], 167),        # Ví dụ 1
    ([1, 5], 10),               # Ví dụ 2
    ([5], 5),                   # 1 quả: 1*5*1
    ([0], 0),                   # 1 quả số 0
    ([0, 0, 0], 0),             # toàn số 0
    ([1, 1, 1], 3),             # mỗi lần nổ được đúng 1 xu
    ([9, 0, 9], 90),            # số 0 ở giữa
    ([100, 100, 100], 1010100), # giá trị lớn nhất theo constraints
]
for nums, expected in tests:
    got = sol.maxCoins(nums)
    print(f"{str(nums):<18} -> {got:>8} (kỳ vọng {expected})")
    assert got == expected
print("Tất cả test của class Solution đều PASS!")

[3, 1, 5, 8]       ->      167 (kỳ vọng 167)
[1, 5]             ->       10 (kỳ vọng 10)
[5]                ->        5 (kỳ vọng 5)
[0]                ->        0 (kỳ vọng 0)
[0, 0, 0]          ->        0 (kỳ vọng 0)
[1, 1, 1]          ->        3 (kỳ vọng 3)
[9, 0, 9]          ->       90 (kỳ vọng 90)
[100, 100, 100]    ->  1010100 (kỳ vọng 1010100)
Tất cả test của class Solution đều PASS!


### Bản nộp thay thế: top-down với `lru_cache`

Ngắn, bám sát công thức nhất. Với `n ≤ 300` độ sâu đệ quy khoảng 300 tầng, vẫn dưới giới hạn mặc định của Python. Mỗi lần gọi `maxCoins` tạo một hàm `dp` mới nên sổ memo không bị "dính" giữa các test.

In [25]:
class SolutionMemo:
    def maxCoins(self, nums: List[int]) -> int:
        arr = [1] + nums + [1]

        @lru_cache(maxsize=None)
        def dp(i: int, j: int) -> int:
            if j - i < 2:
                return 0
            return max(dp(i, k) + arr[i] * arr[k] * arr[j] + dp(k, j) for k in range(i + 1, j))

        return dp(0, len(arr) - 1)

for nums, expected in tests:
    assert SolutionMemo().maxCoins(nums) == expected
print("SolutionMemo PASS toàn bộ", len(tests), "test")

SolutionMemo PASS toàn bộ 8 test


## 10. Test đối chiếu chéo trên input ngẫu nhiên

Các ví dụ viết tay có thể bỏ sót tình huống khó. Ta dùng `random.seed(312)` (cố định để chạy lại ra đúng kết quả) và kiểm tra theo 3 tầng:

1. **n ≤ 7, giá trị 0..9** (nhiều số 0 và số trùng): mọi cách phải khớp **brute force**; thứ tự truy vết đưa vào `simulate_order` phải ra đúng max.
2. **8 ≤ n ≤ 12**: brute force quá chậm, so Cách 2 (memo tuple) với Cách 3 và Cách 4.
3. **13 ≤ n ≤ 80**: so Cách 3, Cách 4, `Solution`, và thứ tự truy vết (mô phỏng lại).

Cuối cùng đo thời gian `Solution` ở kích thước tối đa `n = 300`.

In [26]:
random.seed(312)
small = 0
for _ in range(250):
    n = random.randint(1, 7)
    nums = [random.randint(0, 9) for _ in range(n)]
    expected = max_coins_bruteforce(nums)
    best, order = max_coins_with_order(nums)
    got = {
        "Cách 2 memo tuple": max_coins_subset_memo(nums),
        "Cách 3 top-down": max_coins_topdown(nums),
        "Cách 4 bottom-up": max_coins_dp(nums),
        "Cách 5 truy vết": best,
        "Solution": Solution().maxCoins(nums),
        "SolutionMemo": SolutionMemo().maxCoins(nums),
    }
    for name, value in got.items():
        assert value == expected, (name, nums, value, expected)
    assert simulate_order(nums, order) == expected, (nums, order)
    small += 1
print(f"Tầng 1: {small} bộ n ≤ 7 -> 6 cách đều khớp brute force, thứ tự truy vết mô phỏng ra đúng max")

medium = 0
for _ in range(60):
    nums = [random.randint(0, 100) for _ in range(random.randint(8, 12))]
    assert max_coins_subset_memo(nums) == max_coins_topdown(nums) == max_coins_dp(nums), nums
    medium += 1
print(f"Tầng 2: {medium} bộ 8 ≤ n ≤ 12 -> Cách 2 = Cách 3 = Cách 4")

big = 0
for _ in range(30):
    nums = [random.randint(0, 100) for _ in range(random.randint(13, 80))]
    best, order = max_coins_with_order(nums)
    assert max_coins_topdown(nums) == max_coins_dp(nums) == Solution().maxCoins(nums) == best
    assert simulate_order(nums, order) == best
    big += 1
print(f"Tầng 3: {big} bộ 13 ≤ n ≤ 80 -> Cách 3 = Cách 4 = Solution = mô phỏng thứ tự truy vết")

for label, nums in [("ngẫu nhiên", [random.randint(0, 100) for _ in range(300)]), ("toàn 100", [100] * 300)]:
    t0 = time.perf_counter()
    ans = Solution().maxCoins(nums)
    print(f"n = 300 ({label}): đáp án có {len(str(ans))} chữ số, mất {time.perf_counter() - t0:.2f} giây")

Tầng 1: 250 bộ n ≤ 7 -> 6 cách đều khớp brute force, thứ tự truy vết mô phỏng ra đúng max


Tầng 2: 60 bộ 8 ≤ n ≤ 12 -> Cách 2 = Cách 3 = Cách 4


Tầng 3: 30 bộ 13 ≤ n ≤ 80 -> Cách 3 = Cách 4 = Solution = mô phỏng thứ tự truy vết


n = 300 (ngẫu nhiên): đáp án có 9 chữ số, mất 0.35 giây


n = 300 (toàn 100): đáp án có 9 chữ số, mất 0.45 giây


## 11. Vì sao thuật toán đúng?

**Bất biến (invariant) của bảng:** sau khi điền xong mọi khoảng có độ dài `≤ L`, mỗi ô `dp[i][j]` với `j - i ≤ L` **đúng bằng** "số xu lớn nhất khi nổ hết các quả nằm giữa `i` và `j`, trong khi `i` và `j` còn nguyên".

Giả sử mọi khoảng ngắn hơn `(i, j)` đã đúng. Ta chứng minh `dp[i][j]` cũng đúng bằng hai chiều:

1. **Không thể làm tốt hơn công thức (≤).** Xét **bất kỳ** thứ tự nổ nào của các quả trong `(i, j)`. Nó có đúng **một** quả nổ cuối, gọi là `k`. Trước lúc đó `k` luôn còn sống, nên mỗi quả bên trái `k` chỉ có hàng xóm trong đoạn `[i..k]` → tổng xu của nhóm trái là xu của **một** thứ tự nổ cho bài con `(i, k)`, nên `≤ dp[i][k]`. Tương tự nhóm phải `≤ dp[k][j]`. Quả `k` nổ khi mọi quả ở giữa đã hết → hàng xóm là đúng `i` và `j` → được `arr[i]*arr[k]*arr[j]`. Cộng lại: tổng `≤ dp[i][k] + arr[i]*arr[k]*arr[j] + dp[k][j] ≤ max theo k`.
2. **Công thức đạt được (≥).** Với mỗi `k`, có một thứ tự cụ thể: nổ nhóm trái theo thứ tự tối ưu của `(i, k)`, rồi nhóm phải theo thứ tự tối ưu của `(k, j)`, cuối cùng nổ `k`. Thứ tự này thu **đúng** `dp[i][k] + arr[i]*arr[k]*arr[j] + dp[k][j]`. Lấy `k` tốt nhất → đạt được giá trị max. (Đây cũng chính là thứ tự mà Cách 5 dựng lại.)

Hai chiều cộng lại → `dp[i][j]` đúng. Base case: khoảng rỗng cho 0 xu. Cuối cùng, hai bóng ảo `arr[0] = arr[n+1] = 1` không bao giờ nổ, đúng quy tắc "ngoài biên coi như bóng số 1" của đề → `dp[0][n+1]` là đáp án.

**Vì sao cách "nổ đầu tiên" không qua được chứng minh này?** Ở bước 1, nếu tách theo quả nổ **đầu tiên**, hai nhóm chạm vào nhau ngay sau đó, nên tổng xu **không** viết được thành "xu nhóm trái + xu nhóm phải" — mắt xích "độc lập" bị gãy.

## 12. So sánh các cách

Gọi `n = len(nums)`.

| Cách | Trạng thái | Time | Space | Chạy được tới | Ghi chú |
|---|---|---|---|---|---|
| 1. Brute force (permutations) | một thứ tự | `O(n! * n^2)` | `O(n)` | n ≈ 8 | Dễ nghĩ nhất, làm "trọng tài" để test |
| 2. Đệ quy + memo trên tuple | tập bóng còn lại | `O(2^n * n^2)` | `O(2^n * n)` | n ≈ 18 | Đúng nhờ giữ nguyên cả tuple, nhưng vẫn hàm mũ |
| 3. Interval DP top-down | khoảng mở `(i, j)` | `O(n^3)` | `O(n^2)` + đệ quy `O(n)` | n = 300 ✓ | Dịch thẳng công thức "nổ cuối" |
| 4. Interval DP bottom-up | ô `dp[i][j]` | `O(n^3)` | `O(n^2)` | n = 300 ✓ | **Cách chính**, không đệ quy, in được bảng |
| 5. Bottom-up + truy vết | thêm `choice[i][j]` | `O(n^3)` | `O(n^2)` | n = 300 ✓ | Trả thêm thứ tự nổ tối ưu |

Bước nhảy quan trọng nhất là từ Cách 2 sang Cách 3: **đổi cách nhìn** (nổ đầu tiên → nổ cuối cùng) biến trạng thái từ "tập con bất kỳ" (`2^n`) thành "đoạn liên tiếp" (`n^2`).

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tham lam (greedy) — "cứ nổ quả nhỏ nhất trước"

Trực giác: quả nhỏ "ít giá trị", nổ trước để giữ quả lớn làm hệ số nhân. Nghe hợp lý, và đúng là trong lời giải tối ưu quả lớn thường nổ muộn — nhưng **tham lam theo từng bước không đảm bảo tối ưu toàn cục**. Ngay Ví dụ 1 đã là phản ví dụ.

Trong code dưới, `min(range(len(cur)), key=lambda q: cur[q])` trả về **vị trí** `q` có giá trị `cur[q]` nhỏ nhất (`key` cho `min` biết "so sánh theo cái gì").

In [27]:
def max_coins_greedy_smallest(nums):
    # SAI: luôn nổ quả nhỏ nhất hiện tại.
    cur = list(nums)
    total = 0
    while cur:
        p = min(range(len(cur)), key=lambda q: cur[q])   # vị trí quả nhỏ nhất
        total += coins_at(cur, p)
        cur.pop(p)
    return total

for nums in ([3, 1, 5, 8], [1, 2, 9, 2, 1]):
    print(f"{nums}: greedy = {max_coins_greedy_smallest(nums)}, đúng = {max_coins_dp(nums)}")

[3, 1, 5, 8]: greedy = 78, đúng = 167
[1, 2, 9, 2, 1]: greedy = 49, đúng = 63


Với `[3, 1, 5, 8]`, greedy nổ `1` → `3` → `5` → `8` được `15 + 15 + 40 + 8 = 78`, kém xa 167: nó nổ quả `3` quá sớm trong khi `3` là hệ số nhân rất quý cho quả `5`. **Sửa:** không có quy tắc tham lam đơn giản nào đúng cho bài này — phải xét mọi lựa chọn `k` bằng DP.

### Lỗi 2: Chia theo quả nổ ĐẦU TIÊN rồi giải hai nửa "độc lập"

Đây là lỗi "nghe rất DP" mà mục 5.1 đã phân tích. Code sai thường trông thế này: chọn `k` nổ đầu tiên với hàng xóm **ban đầu** `k-1`, `k+1`, rồi cộng kết quả của hai nửa giải riêng. Nó sai theo **cả hai chiều**: có lúc ra **thấp hơn** đáp án (bỏ lỡ cách hai nửa phối hợp), có lúc ra **cao hơn** đáp án — tức là "bịa" ra số xu không thể đạt được, vì nửa trái và nửa phải đều dùng quả `k` (đã nổ!) làm hàng xóm.

In [28]:
def max_coins_wrong_first_burst(nums):
    n = len(nums)

    def val(t):                          # giá trị quả ở chỉ số gốc t, ngoài biên = 1
        return nums[t] if 0 <= t < n else 1

    @lru_cache(maxsize=None)
    def f(lo, hi):                       # "giải độc lập" đoạn đóng nums[lo..hi]
        if lo > hi:
            return 0
        best = 0
        for k in range(lo, hi + 1):      # SAI: coi k là quả nổ ĐẦU TIÊN
            best = max(best, val(k - 1) * nums[k] * val(k + 1) + f(lo, k - 1) + f(k + 1, hi))
        return best

    return f(0, n - 1)

for nums in ([3, 1, 5, 8], [3, 5, 4]):
    print(f"{nums}: sai = {max_coins_wrong_first_burst(nums)}, đúng = {max_coins_dp(nums)}")

[3, 1, 5, 8]: sai = 98, đúng = 167
[3, 5, 4]: sai = 95, đúng = 76


Với `[3, 5, 4]`, code sai ra 95 > 76: nó nổ `5` trước (`3*5*4 = 60`), rồi tính nửa trái `[3]` với hàng xóm phải là `5` (`1*3*5 = 15`) và nửa phải `[4]` với hàng xóm trái là `5` (`5*4*1 = 20`) — nhưng quả `5` **đã nổ** từ trước! **Sửa:** chọn `k` là quả nổ **cuối cùng** trong khoảng mở `(i, j)` (Cách 3/4).

### Lỗi 3: Sai thứ tự vòng lặp bottom-up

Viết vòng ngoài là `i` **tăng dần** trông tự nhiên, nhưng khi tính `dp[0][j]` ta cần `dp[k][j]` với `k > 0` — hàng `k` **chưa được tính** (vẫn là 0). Code không báo lỗi gì, chỉ âm thầm ra kết quả sai.

In [29]:
def max_coins_wrong_loop_order(nums):
    arr = [1] + nums + [1]
    m = len(arr)
    dp = [[0] * m for _ in range(m)]
    for i in range(m):                       # SAI: i tăng dần -> dp[k][j] (k > i) chưa có
        for j in range(i + 2, m):
            for k in range(i + 1, j):
                dp[i][j] = max(dp[i][j], dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j])
    return dp[0][m - 1]

print("Thứ tự vòng lặp sai:", max_coins_wrong_loop_order([3, 1, 5, 8]), "| đúng:", max_coins_dp([3, 1, 5, 8]))

Thứ tự vòng lặp sai: 63 | đúng: 167


Ra 63 thay vì 167, vì mọi lần đọc `dp[k][j]` khi tính hàng 0 đều đọc phải số 0. **Sửa:** vòng ngoài theo **độ dài khoảng tăng dần** như Cách 4. (Còn một thứ tự vòng lặp khác cũng đúng — Bài 4 sẽ để bạn tự tìm và giải thích.) Mẹo kiểm tra: với mỗi ô, tự hỏi "các ô nó đọc đã được tính **trước** nó chưa?".

### Lỗi 4: Quên đệm biên, hoặc dùng `nums` thay vì `arr` trong công thức

Nếu viết `nums[i] * nums[k] * nums[j]` trong khi `i, j, k` là chỉ số của mảng **đã đệm**, mọi chỉ số bị lệch 1 → kết quả sai hoặc `IndexError` ở `j = n + 1`. Quy tắc: **đã đệm thì chỉ dùng `arr`**, và chỉ đổi về chỉ số gốc (`k - 1`) khi cần in thứ tự nổ.

### Lỗi 5: Nhầm khoảng MỞ với khoảng ĐÓNG

Trong notebook này `dp(i, j)` là khoảng **mở**: `i`, `j` là tường, **không** nổ. Có người định nghĩa khoảng **đóng** `[i..j]` (nổ cả `i` và `j`) — cũng được, nhưng khi đó quả nổ cuối có hàng xóm là `arr[i-1]` và `arr[j+1]`. Lỗi là **trộn** hai quy ước: định nghĩa đóng mà nhân `arr[i] * arr[k] * arr[j]`, hoặc cho `k` chạy `range(i, j + 1)` với định nghĩa mở. Hãy viết định nghĩa ra giấy trước khi code.

### Lỗi 6: `max()` trên dãy rỗng

Viết gọn `return max(... for k in range(i + 1, j))` mà quên chặn base case `j - i < 2` → khi khoảng rỗng, `range` rỗng và Python báo `ValueError`. **Sửa:** đặt base case trước, hoặc dùng `max(..., default=0)`:

```python
max(x for x in range(3, 3))               # ValueError: range rỗng, không có gì để lấy max
max((x for x in range(3, 3)), default=0)  # 0
```

### Lỗi 7: Ba lỗi "Python" đã gặp ở phần kiến thức nền

- Dùng **`list`** làm tham số cho hàm có `@lru_cache` → `TypeError: unhashable type: 'list'` (mục 4.3). Sửa: đổi sang `tuple`, hoặc tốt hơn là dùng trạng thái `(i, j)`.
- Tạo bảng bằng `[[0] * m] * m` → các hàng dùng chung một list (mục 7). Sửa: `[[0] * m for _ in range(m)]`.
- Lấy hàng xóm bằng `cur[p - 1]` khi `p = 0` → Python lặng lẽ lấy phần tử cuối (mục 2.2). Sửa: kiểm tra `p > 0`, hoặc đệm biên để khỏi phải kiểm tra.

### Lỗi 8: Truy vết bị `None` khi mọi xu đều bằng 0

Lưu `choice` với điều kiện `coins > best` (khởi tạo `best = 0`) thì với `nums = [0, 0]` không lựa chọn nào "lớn hơn 0" → `choice` vẫn là `None` → hàm dựng thứ tự bị lỗi. Sửa: `choice[i][j] is None or coins > dp[i][j]` (mục 8).

## 14. Mẹo nhận dạng pattern

- **Dấu hiệu đề:** phần tử bị **xóa / nổ / gộp / cắt từng cái một**, chi phí (hoặc điểm) mỗi bước phụ thuộc vào **hàng xóm hiện tại**, và ta được chọn **thứ tự** → nghĩ ngay tới **interval DP**.
- **Dấu hiệu constraints:** `n` cỡ 100–500 → gợi ý `O(n^3)`; `n ≤ 20` → có thể bitmask/tập con (như Cách 2); `n ≤ 10` → brute force.
- **Khi thao tác ĐẦU TIÊN làm hai phần dính vào nhau** (như nổ bóng) → hãy thử nghĩ theo thao tác **CUỐI CÙNG** / phần tử **sống lâu nhất**: nó là vách ngăn tách bài thành hai bài con độc lập.
- **Khi thao tác đầu tiên tách hẳn bài thành hai phần rời nhau** (như cắt một cây gậy — Bài 5) → nghĩ theo thao tác **đầu tiên** là được.
- **Đệm phần tử biên (sentinel)** như `[1] + nums + [1]` giúp bỏ hết các `if` kiểm tra mép mảng.
- **Khuôn interval DP:** `dp[i][j]` từ `dp[i][k]` và `dp[k][j]`; điền theo **độ dài tăng dần**; đáp án ở góc `dp[0][m-1]`.

## 15. Biến thể và bài liên quan

- **1039. Minimum Score Triangulation of Polygon** — công thức gần như y hệt: `dp[i][j] = min over k (dp[i][k] + v[i]*v[k]*v[j] + dp[k][j])`.
- **1547. Minimum Cost to Cut a Stick** — interval DP, nhưng nghĩ theo nhát cắt **đầu tiên** (Bài 5).
- **1000. Minimum Cost to Merge Stones** — interval DP có thêm điều kiện gộp đúng `k` đống.
- **375. Guess Number Higher or Lower II** — chọn số đoán `k` chia khoảng `[i..j]` thành hai phía.
- **1130. Minimum Cost Tree From Leaf Values**, **516. Longest Palindromic Subsequence** — interval DP dễ hơn để luyện tay.
- **664. Strange Printer**, **546. Remove Boxes** — interval DP khó hơn (cần thêm chiều trạng thái).
- **Matrix Chain Multiplication** (nhân chuỗi ma trận) — bài kinh điển trong sách giáo khoa, cùng khuôn `dp[i][k] + dp[k][j] + chi phí ghép`.

### Mẹo phụ: có thể bỏ hết các quả số 0 trước khi DP

Một quả số 0 nổ luôn cho 0 xu, và khi nó còn sống thì quả nào cạnh nó nổ cũng được 0 xu. Ta có thể chứng minh `maxCoins(nums) == maxCoins(nums không có số 0)`:

- **(≥)** Nổ hết các quả 0 trước (mỗi quả cho 0 xu), phần còn lại chính là mảng đã lọc → đạt được đáp án của mảng lọc.
- **(≤)** Lấy một thứ tự bất kỳ trên mảng gốc và giữ nguyên thứ tự đó cho các quả khác 0 trên mảng lọc. Mỗi lần nổ một quả khác 0: nếu hàng xóm của nó (trên mảng gốc) khác 0 thì trên mảng lọc hàng xóm giống hệt; nếu hàng xóm là 0 thì xu trên mảng gốc bằng 0, không lớn hơn xu trên mảng lọc. Vậy mảng gốc không thể hơn mảng lọc.

Mẹo này làm `n` nhỏ đi khi có nhiều số 0 (không đổi Big-O trường hợp xấu nhất). Kiểm chứng:

In [30]:
def max_coins_skip_zeros(nums):
    return max_coins_dp([x for x in nums if x != 0])

random.seed(7)
for _ in range(200):
    nums = [random.choice([0, 0, 1, 2, 5, 9]) for _ in range(random.randint(1, 9))]
    assert max_coins_skip_zeros(nums) == max_coins_dp(nums), nums
print("Bỏ số 0 trước khi DP: 200 bộ test ngẫu nhiên đều cho cùng đáp án")
print("Ví dụ: [9, 0, 9] ->", max_coins_dp([9, 0, 9]), "=", max_coins_skip_zeros([9, 0, 9]), "(như [9, 9])")

Bỏ số 0 trước khi DP: 200 bộ test ngẫu nhiên đều cho cùng đáp án
Ví dụ: [9, 0, 9] -> 90 = 90 (như [9, 9])


## 16. Bài tập vận dụng

Làm theo thứ tự; mỗi bài có gợi ý 3 mức ở mục 17 và đáp án ở mục 18 (cuối notebook). Hãy tự làm trước khi xem.

### Bài 1 — Đoán output (tính tay, không chạy code)

Cho `nums = [2, 4, 3]`. Tính số xu của 3 thứ tự nổ (theo chỉ số gốc):

- (a) `[0, 1, 2]`
- (b) `[1, 0, 2]`
- (c) `[2, 1, 0]`

Câu hỏi phụ: trong **mọi** thứ tự, quả nổ cuối cùng luôn được bao nhiêu xu? Vì sao?

### Bài 2 — Tính tay bảng `dp`

Vẫn `nums = [2, 4, 3]`, `arr = [1, 2, 4, 3, 1]`. Điền tay các ô theo thứ tự độ dài tăng dần: `dp[0][2]`, `dp[1][3]`, `dp[2][4]`, rồi `dp[0][3]`, `dp[1][4]`, cuối cùng `dp[0][4]`. Với mỗi ô ghi rõ giá trị của từng lựa chọn `k` và `k` được chọn. Sau đó dùng các `k` đã chọn để **dựng lại thứ tự nổ** tối ưu, và so với câu trả lời Bài 1.

### Bài 3 — Điền TODO: interval DP top-down

Hoàn thiện hàm dưới đây theo đúng công thức "quả nổ cuối cùng".

In [31]:
def exercise_max_coins_memo(nums):
    arr = [1] + nums + [1]

    @lru_cache(maxsize=None)
    def dp(i, j):
        # TODO 1: base case — nếu không còn quả nào nằm giữa i và j thì trả về bao nhiêu?
        # TODO 2: thử mọi k trong khoảng MỞ (i, j) làm quả nổ CUỐI CÙNG
        # TODO 3: xu = dp(i, k) + ??? + dp(k, j); giữ giá trị lớn nhất
        pass

    return dp(0, len(arr) - 1)

# Tự làm xong rồi mới mở test:
# assert exercise_max_coins_memo([3, 1, 5, 8]) == 167
# assert exercise_max_coins_memo([1, 5]) == 10
# assert exercise_max_coins_memo([2, 4, 3]) == 33
# assert exercise_max_coins_memo([0]) == 0

### Bài 4 — Tự viết lại bottom-up với thứ tự vòng lặp KHÁC

Viết lại Cách 4 **không** dùng vòng `length`, mà dùng:

- vòng ngoài: `i` chạy **giảm dần** từ `m - 3` về `0`;
- vòng trong: `j` chạy **tăng dần** từ `i + 2` tới `m - 1`.

Trước khi code, hãy tự giải thích bằng lời: khi tính `dp[i][j]` theo thứ tự này, vì sao `dp[i][k]` và `dp[k][j]` đều **đã có sẵn**? (So sánh với Lỗi 3, nơi `i` chạy tăng dần.)

In [32]:
def exercise_max_coins_dp_reverse(nums):
    # TODO 1: arr = [1] + nums + [1]; m = len(arr); tạo bảng dp m x m toàn 0 (tạo ĐÚNG cách!)
    # TODO 2: for i in range(m - 3, -1, -1):
    # TODO 3:     for j in range(i + 2, m):
    # TODO 4:         dp[i][j] = max theo k trong (i, j) của dp[i][k] + arr[i]*arr[k]*arr[j] + dp[k][j]
    # TODO 5: return dp[0][m - 1]
    pass

# Gợi ý test:
# assert exercise_max_coins_dp_reverse([3, 1, 5, 8]) == 167
# assert exercise_max_coins_dp_reverse([1, 5]) == 10
# assert exercise_max_coins_dp_reverse([5]) == 5
# assert exercise_max_coins_dp_reverse([9, 0, 9]) == 90

### Bài 5 — Biến thể: LeetCode 1547. Minimum Cost to Cut a Stick

Cho một cây gậy gỗ dài `n` đơn vị, các vị trí được đánh dấu từ `0` đến `n`. Mảng số nguyên `cuts[i]` là vị trí cần cắt. Bạn được thực hiện các nhát cắt **theo thứ tự tùy ý**. Chi phí của một nhát cắt là **độ dài của khúc gậy đang bị cắt**; tổng chi phí là tổng chi phí mọi nhát cắt. Khi cắt, khúc gậy tách thành hai khúc nhỏ hơn (tổng độ dài bằng khúc ban đầu). Trả về **tổng chi phí nhỏ nhất** để thực hiện hết các nhát cắt.

```text
Ví dụ 1: n = 7, cuts = [1,3,4,5]      -> 16
Ví dụ 2: n = 9, cuts = [5,6,1,4,2]    -> 22
```

Constraints: `2 <= n <= 10^6`; `1 <= cuts.length <= min(n - 1, 100)`; `1 <= cuts[i] <= n - 1`; các phần tử của `cuts` khác nhau. Signature trên LeetCode: `def minCost(self, n: int, cuts: List[int]) -> int`.

Điểm thú vị: bài này cũng là interval DP, nhưng ở đây nghĩ theo nhát cắt **đầu tiên** lại hợp lý. Hãy tự giải thích vì sao (so với bóng nổ).

In [33]:
def exercise_min_cost_cut(n, cuts):
    # TODO 1: c = [0] + sorted(cuts) + [n]  (đệm hai đầu gậy, giống đệm bóng ảo)
    # TODO 2: dp[i][j] = chi phí nhỏ nhất để cắt hết các điểm nằm GIỮA c[i] và c[j]
    # TODO 3: dp[i][j] = (c[j] - c[i]) + min theo k trong (i, j) của dp[i][k] + dp[k][j]
    # TODO 4: điền theo độ dài khoảng tăng dần; khoảng không có điểm cắt nào -> 0
    pass

# Gợi ý test:
# assert exercise_min_cost_cut(7, [1, 3, 4, 5]) == 16
# assert exercise_min_cost_cut(9, [5, 6, 1, 4, 2]) == 22
# assert exercise_min_cost_cut(2, [1]) == 2

### Bài 6 — Mini challenge: đếm số thứ tự nổ đạt tối đa

Viết `exercise_count_best_orders(nums)` trả về **số thứ tự nổ khác nhau** (trong `n!` thứ tự) cho đúng số xu lớn nhất. Ví dụ `[3, 1, 5, 8]` chỉ có 1 thứ tự (bảng xếp hạng ở Cách 1), còn `[1, 1, 1]` thì cả 6 thứ tự đều tối ưu.

Yêu cầu: **không** duyệt `n!` thứ tự; phải chạy được với `n` cỡ 50 trở lên.

Công cụ mới cần dùng: `math.comb(a, b)` = số cách **chọn** `b` phần tử từ `a` phần tử (tổ hợp). Ví dụ `comb(4, 2) = 6`: có 6 cách chọn 2 trong 4 vị trí. Cell TODO dưới in thử vài giá trị để bạn làm quen.

In [34]:
from math import comb
print("comb(4, 2) =", comb(4, 2), "| comb(3, 0) =", comb(3, 0), "| comb(5, 5) =", comb(5, 5))

def exercise_count_best_orders(nums):
    # TODO 1: điền bảng dp như Cách 4 (dp[i][j] = số xu lớn nhất trong khoảng mở (i, j))
    # TODO 2: cnt[i][j] = số thứ tự nổ các quả trong (i, j) đạt đúng dp[i][j]; khoảng rỗng -> 1
    # TODO 3: với MỌI k tối ưu (giá trị đúng bằng dp[i][j]), cộng thêm
    #         comb(a + b, a) * cnt[i][k] * cnt[k][j], với a, b = số quả của nhóm trái, nhóm phải
    # TODO 4: return cnt[0][m - 1]
    pass

# Gợi ý test:
# assert exercise_count_best_orders([3, 1, 5, 8]) == 1
# assert exercise_count_best_orders([1, 1, 1]) == 6
# assert exercise_count_best_orders([0, 0]) == 2
# assert exercise_count_best_orders([1, 2, 9, 2, 1]) == 6
# assert exercise_count_best_orders([7, 9, 8, 0, 7]) == 2

comb(4, 2) = 6 | comb(3, 0) = 1 | comb(5, 5) = 1


## 17. Gợi ý theo mức

Chỉ mở mức tiếp theo khi mức trước chưa đủ.

**Bài 1**
- Mức 1: Làm đúng như bảng tính tay ở mục 1: mỗi bước ghi "mảng còn lại", hàng xóm trái/phải **hiện tại**.
- Mức 2: Nổ quả ở mép thì hàng xóm bên ngoài là bóng ảo 1.
- Mức 3: (b) bắt đầu bằng `2*4*3 = 24`, còn `[2, 3]`.

**Bài 2**
- Mức 1: Khoảng chứa 1 quả chỉ có một lựa chọn: `dp[i][i+2] = arr[i]*arr[i+1]*arr[i+2]`.
- Mức 2: `dp[0][3]` có 2 lựa chọn `k = 1, 2`; tường là `arr[0] = 1` và `arr[3] = 3`.
- Mức 3: Thứ tự nổ = đọc cây từ lá lên gốc: nhóm trái, nhóm phải, rồi mới tới `k` của ô `dp[0][4]`.

**Bài 3**
- Mức 1: Base case: `j - i < 2` nghĩa là không có quả nào giữa → 0 xu.
- Mức 2: `for k in range(i + 1, j)` — không gồm `i` và `j` vì chúng là tường.
- Mức 3: Phần `???` là xu của quả `k` khi nổ cuối: hàng xóm lúc đó là hai bức tường.

**Bài 4**
- Mức 1: Ô `dp[i][j]` đọc các ô **cùng hàng `i`** nhưng cột nhỏ hơn `j`, và các ô **cùng cột `j`** nhưng hàng lớn hơn `i`.
- Mức 2: `i` giảm dần → mọi hàng `k > i` đã điền xong; `j` tăng dần → trong hàng `i`, các cột `k < j` đã điền xong.
- Mức 3: `range(m - 3, -1, -1)` cho `i = m-3, ..., 1, 0` (hàng `m-2`, `m-1` không có ô nào cần tính).

**Bài 5**
- Mức 1: Sắp xếp `cuts`, đệm `0` và `n` hai đầu → `c`. Bài con: khúc gậy từ `c[i]` tới `c[j]`, cắt hết các điểm `c[k]` với `i < k < j`.
- Mức 2: Nhát cắt **đầu tiên** trên khúc `[c[i], c[j]]` tốn `c[j] - c[i]` bất kể cắt ở đâu; sau đó hai khúc gậy **rời nhau thật sự** (không như bóng nổ, hai khúc không bao giờ ảnh hưởng nhau nữa) → hai bài con độc lập.
- Mức 3: `dp[i][j] = c[j] - c[i] + min(dp[i][k] + dp[k][j] for k in range(i + 1, j))`, với `dp[i][i+1] = 0`.

**Bài 6**
- Mức 1: Mỗi thứ tự tối ưu có một quả nổ cuối `k`. Theo chứng minh ở mục 11, nó tối ưu khi và chỉ khi `k` là một lựa chọn tối ưu **và** nhóm trái, nhóm phải đều nổ theo thứ tự tối ưu của riêng chúng. Có thể có **nhiều** `k` hòa nhau.
- Mức 2: Nhóm trái có `a` quả, nhóm phải có `b` quả. Các lần nổ của hai nhóm được **xen kẽ tùy ý** (giữ nguyên thứ tự bên trong mỗi nhóm): chọn `a` vị trí trong `a + b` lượt cho nhóm trái → `comb(a + b, a)` cách.
- Mức 3: `a = k - i - 1`, `b = j - k - 1`; `cnt[i][j] = tổng theo k tối ưu của comb(a + b, a) * cnt[i][k] * cnt[k][j]`; tạo `cnt` toàn 1 để khoảng rỗng có `cnt = 1`.

## 18. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

| Thứ tự | Bước 1 | Bước 2 | Bước 3 | Tổng |
|---|---|---|---|---:|
| (a) `[0, 1, 2]` | nổ 2: `1*2*4 = 8`, còn `[4, 3]` | nổ 4: `1*4*3 = 12`, còn `[3]` | nổ 3: `1*3*1 = 3` | **23** |
| (b) `[1, 0, 2]` | nổ 4: `2*4*3 = 24`, còn `[2, 3]` | nổ 2: `1*2*3 = 6`, còn `[3]` | nổ 3: `1*3*1 = 3` | **33** |
| (c) `[2, 1, 0]` | nổ 3: `4*3*1 = 12`, còn `[2, 4]` | nổ 4: `2*4*1 = 8`, còn `[2]` | nổ 2: `1*2*1 = 2` | **22** |

Câu hỏi phụ: quả nổ cuối cùng luôn được `1 * x * 1 = x` xu (với `x` là số của nó), vì lúc đó mọi quả khác đã nổ, hai bên chỉ còn bóng ảo 1. Đây chính là hạt giống của ý tưởng "quả nổ cuối cùng".

### Đáp án Bài 2

`arr = [1, 2, 4, 3, 1]`:

| Ô | Lựa chọn `k` → giá trị | Chọn | `dp` |
|---|---|:---:|---:|
| `dp[0][2]` | `k=1`: `1*2*4 = 8` | 1 | 8 |
| `dp[1][3]` | `k=2`: `2*4*3 = 24` | 2 | 24 |
| `dp[2][4]` | `k=3`: `4*3*1 = 12` | 3 | 12 |
| `dp[0][3]` | `k=1`: `0 + 1*2*3 + 24 = 30`; `k=2`: `8 + 1*4*3 + 0 = 20` | 1 | 30 |
| `dp[1][4]` | `k=2`: `0 + 2*4*1 + 12 = 20`; `k=3`: `24 + 2*3*1 + 0 = 30` | 3 | 30 |
| `dp[0][4]` | `k=1`: `0 + 1*2*1 + 30 = 32`; `k=2`: `8 + 1*4*1 + 12 = 24`; `k=3`: `30 + 1*3*1 + 0 = 33` | 3 | **33** |

Dựng thứ tự: ô `dp[0][4]` chọn `k = 3` (quả `3`, chỉ số gốc 2) nổ **cuối**; nhóm trái `dp[0][3]` chọn `k = 1` (quả `2`, chỉ số gốc 0) nổ cuối trong nhóm; còn lại `dp[1][3]` là quả `4` (chỉ số gốc 1) nổ đầu tiên. Thứ tự: `[1, 0, 2]` → 33 xu, đúng là thứ tự (b) của Bài 1. Cell dưới kiểm tra cả hai bài bằng code.

In [35]:
# ===== Kiểm tra đáp án Bài 1 và Bài 2 =====
nums = [2, 4, 3]
assert simulate_order(nums, [0, 1, 2]) == 23
assert simulate_order(nums, [1, 0, 2]) == 33
assert simulate_order(nums, [2, 1, 0]) == 22
assert max_coins_bruteforce(nums) == 33

arr, dp, choice = build_dp_tables(nums)
assert (dp[0][2], dp[1][3], dp[2][4]) == (8, 24, 12)
assert (dp[0][3], dp[1][4], dp[0][4]) == (30, 30, 33)
assert (choice[0][3], choice[1][4], choice[0][4]) == (1, 3, 3)
assert max_coins_with_order(nums) == (33, [1, 0, 2])
print("Đáp án Bài 1 và Bài 2: PASS")


# ===== Đáp án Bài 3: interval DP top-down =====
def answer_max_coins_memo(nums):
    arr = [1] + nums + [1]

    @lru_cache(maxsize=None)
    def dp(i, j):
        if j - i < 2:                                    # TODO 1: không có quả nào ở giữa
            return 0
        best = 0
        for k in range(i + 1, j):                        # TODO 2: k nổ cuối cùng
            best = max(best, dp(i, k) + arr[i] * arr[k] * arr[j] + dp(k, j))   # TODO 3
        return best

    return dp(0, len(arr) - 1)


# ===== Đáp án Bài 4: bottom-up, i giảm dần, j tăng dần =====
def answer_max_coins_dp_reverse(nums):
    arr = [1] + nums + [1]
    m = len(arr)
    dp = [[0] * m for _ in range(m)]
    for i in range(m - 3, -1, -1):          # hàng k > i đã điền xong trước
        for j in range(i + 2, m):           # trong hàng i, các cột k < j đã điền xong trước
            dp[i][j] = max(dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j] for k in range(i + 1, j))
    return dp[0][m - 1]


for f in (answer_max_coins_memo, answer_max_coins_dp_reverse):
    assert f([3, 1, 5, 8]) == 167
    assert f([1, 5]) == 10
    assert f([2, 4, 3]) == 33
    assert f([0]) == 0
    assert f([5]) == 5
    assert f([9, 0, 9]) == 90

random.seed(2024)
for _ in range(100):
    nums = [random.randint(0, 100) for _ in range(random.randint(1, 40))]
    expected = max_coins_dp(nums)
    assert answer_max_coins_memo(nums) == expected
    assert answer_max_coins_dp_reverse(nums) == expected
print("Đáp án Bài 3 và Bài 4: PASS (ví dụ + 100 bộ ngẫu nhiên)")

Đáp án Bài 1 và Bài 2: PASS


Đáp án Bài 3 và Bài 4: PASS (ví dụ + 100 bộ ngẫu nhiên)


In [36]:
# ===== Đáp án Bài 5: Minimum Cost to Cut a Stick (LeetCode 1547) =====
def answer_min_cost_cut(n, cuts):
    c = [0] + sorted(cuts) + [n]
    m = len(c)
    dp = [[0] * m for _ in range(m)]
    for length in range(2, m):
        for i in range(m - length):
            j = i + length
            dp[i][j] = c[j] - c[i] + min(dp[i][k] + dp[k][j] for k in range(i + 1, j))
    return dp[0][m - 1]

assert answer_min_cost_cut(7, [1, 3, 4, 5]) == 16
assert answer_min_cost_cut(9, [5, 6, 1, 4, 2]) == 22
assert answer_min_cost_cut(2, [1]) == 2

def brute_min_cost_cut(n, cuts):      # thử mọi thứ tự cắt để kiểm chứng
    best = None
    for order in permutations(cuts):
        pieces, cost = [(0, n)], 0
        for x in order:
            for idx, (a, b) in enumerate(pieces):
                if a < x < b:
                    cost += b - a
                    pieces[idx:idx + 1] = [(a, x), (x, b)]
                    break
        best = cost if best is None else min(best, cost)
    return best

random.seed(1547)
for _ in range(100):
    n = random.randint(2, 20)
    cuts = random.sample(range(1, n), random.randint(1, min(n - 1, 6)))
    assert answer_min_cost_cut(n, cuts) == brute_min_cost_cut(n, cuts), (n, cuts)
print("Đáp án Bài 5: PASS (ví dụ đề + 100 bộ so với brute force)")


# ===== Đáp án Bài 6: đếm số thứ tự nổ đạt tối đa =====
def answer_count_best_orders(nums):
    arr = [1] + nums + [1]
    m = len(arr)
    dp = [[0] * m for _ in range(m)]
    cnt = [[1] * m for _ in range(m)]            # khoảng rỗng: đúng 1 cách (không làm gì)
    for length in range(2, m):
        for i in range(m - length):
            j = i + length
            vals = {k: dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j] for k in range(i + 1, j)}
            dp[i][j] = max(vals.values())
            total = 0
            for k, v in vals.items():
                if v == dp[i][j]:                 # mọi k tối ưu (có thể hòa nhau)
                    a, b = k - i - 1, j - k - 1   # số quả nhóm trái, nhóm phải
                    total += comb(a + b, a) * cnt[i][k] * cnt[k][j]
            cnt[i][j] = total
    return cnt[0][m - 1]

assert answer_count_best_orders([3, 1, 5, 8]) == 1
assert answer_count_best_orders([1, 1, 1]) == 6
assert answer_count_best_orders([0, 0]) == 2
assert answer_count_best_orders([1, 2, 9, 2, 1]) == 6
assert answer_count_best_orders([7, 9, 8, 0, 7]) == 2

random.seed(6)
for _ in range(150):
    nums = [random.randint(0, 3) for _ in range(random.randint(1, 6))]
    best = max_coins_bruteforce(nums)
    brute_count = sum(1 for o in permutations(range(len(nums))) if simulate_order(nums, o) == best)
    assert answer_count_best_orders(nums) == brute_count, nums
print("Đáp án Bài 6: PASS (ví dụ + 150 bộ so với đếm brute force)")
assert answer_count_best_orders([0] * 50) == factorial(50)
print("[0] * 50: số thứ tự tối ưu đúng bằng 50! (mọi thứ tự đều cho 0 xu) — chạy tức thì, không cần duyệt 50!")

Đáp án Bài 5: PASS (ví dụ đề + 100 bộ so với brute force)


Đáp án Bài 6: PASS (ví dụ + 150 bộ so với đếm brute force)
[0] * 50: số thứ tự tối ưu đúng bằng 50! (mọi thứ tự đều cho 0 xu) — chạy tức thì, không cần duyệt 50!


## 19. Template ngắn nên nhớ

```python
class Solution:
    def maxCoins(self, nums: List[int]) -> int:
        arr = [1] + nums + [1]                   # 1) đệm 2 bóng ảo làm tường
        m = len(arr)
        dp = [[0] * m for _ in range(m)]         # 2) dp[i][j]: khoảng MỞ (i, j), i và j không nổ
        for length in range(2, m):               # 3) điền theo độ dài khoảng tăng dần
            for i in range(m - length):
                j = i + length
                dp[i][j] = max(dp[i][k] + arr[i] * arr[k] * arr[j] + dp[k][j]   # 4) k nổ CUỐI
                               for k in range(i + 1, j))
        return dp[0][m - 1]                      # 5) đáp án ở góc trên bên phải
```

**Câu thần chú:** *Đệm 1 hai đầu → `dp(i, j)` là khoảng mở có hai tường → chọn `k` nổ CUỐI → `dp(i,k) + arr[i]·arr[k]·arr[j] + dp(k,j)` → điền theo độ dài tăng dần → đáp án `dp[0][n+1]`.*

## 20. Checklist tự kiểm tra trước khi submit

- [ ] Đã đệm `arr = [1] + nums + [1]` và **chỉ dùng `arr`** trong công thức (không lẫn `nums`)?
- [ ] Định nghĩa `dp[i][j]` là khoảng **mở**: `i`, `j` là tường, không nổ trong bài con?
- [ ] `k` chạy `range(i + 1, j)` — không gồm `i`, `j`?
- [ ] Xu của `k` là `arr[i] * arr[k] * arr[j]` (hàng xóm là **hai tường**, không phải `arr[k-1]`, `arr[k+1]`)?
- [ ] Thứ tự vòng lặp: theo độ dài tăng dần (hoặc `i` giảm dần, `j` tăng dần)? Mọi ô được đọc đã tính trước chưa?
- [ ] Tạo bảng bằng `[[0] * m for _ in range(m)]`, không phải `[[0] * m] * m`?
- [ ] Không gọi `max()` trên `range` rỗng (base case `j - i < 2` → 0)?
- [ ] Trả về `dp[0][m - 1]` (không phải `dp[1][n]` hay `dp[0][n]`)?
- [ ] Đã test: hai ví dụ của đề, `n = 1`, mảng có số `0`, và đối chiếu với brute force cho `n` nhỏ?
- [ ] Ước lượng thời gian: `n = 300` → khoảng 4,5 triệu phép thử `k` → ổn với Python.